# 00 · FT-Transformer Architecture Variants (Home Credit)

This notebook benchmarks **nine different FT-Transformer modifications** on the Home Credit Default Risk task. All variants share the same data, preprocessing, training loop and hyperparameters (`d_token=64`, `n_blocks=3`, `n_heads=4`, `ffn_d_hidden_multiplier=2`), so their validation/test AUCs are directly comparable.

**Variants**:

| Section | Variant                              | Where SwiGLU / change is applied |
|---------|--------------------------------------|----------------------------------|
| 5       | SwiGLU in numerical embeddings       | Cont. feature embedding layer    |
| 6       | SwiGLU in attention + FFN            | Attention block + FFN            |
| 7       | Cross-feature SwiGLU tokenizer       | Cont. embedding (cross-feature)  |
| 8       | Quantile PLE embedding               | Cont. feature embedding layer    |
| 9       | QPLE + Linear                        | Cont. feature embedding layer    |
| 10      | QPLE + Linear + Periodic             | Cont. feature embedding layer    |
| 11      | Lion optimizer (vanilla FTT)         | Optimizer                        |
| 12      | Sparsemax attention                  | Attention softmax                |
| 13      | Pairwise AUC loss                    | Loss function                    |
| 14      | MLP-based contextual embedding       | Cont. feature embedding layer    |

## 1. Setup

In [ ]:
# %pip install rtdl_revisiting_models -q
%pip install delu -q
%pip install optuna -q

In [ ]:
import math
import random
import warnings
import json
import os
import itertools
import typing
from collections import OrderedDict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Literal, Optional, Tuple, cast

import numpy as np
import pandas as pd
import scipy.special

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim
from torch import Tensor
from torch.nn.parameter import Parameter
from torch.utils.data import Dataset, DataLoader, TensorDataset

import sklearn.datasets
import sklearn.metrics
import sklearn.model_selection
import sklearn.preprocessing
import sklearn.tree as sklearn_tree
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, QuantileTransformer
from sklearn.metrics import roc_auc_score
from sklearn.isotonic import IsotonicRegression

import optuna
import delu
from tqdm.std import tqdm
from tqdm import tqdm as _tqdm  # alias for cells that use it directly

warnings.filterwarnings("ignore")
pd.options.display.max_rows = 200
pd.options.display.max_columns = 200

RANDOM_SEED = 42


def seed_everything(seed: int = RANDOM_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# --- Optional: Google Colab Drive mount ---
# Uncomment the three lines below if you're running on Colab and want to read
# data from / write artifacts to your Drive. Skip on local, server, or Kaggle
# runs. The next cell auto-routes through DRIVE_ROOT when defined.
#
# from google.colab import drive
# drive.mount("/content/drive")
# DRIVE_ROOT = "/content/drive/MyDrive/ft-transformer-credit-risk-study"

# When running locally, repo root is one directory up (notebook is in
# notebooks/<dataset>/). On Colab with the cell above uncommented, DRIVE_ROOT
# takes precedence.
_BASE = globals().get("DRIVE_ROOT", "..")
DATA_PATH      = f"{_BASE}/data/preprocessed_data_home_credit.parquet.gzip"
ARTIFACTS_DIR  = Path(f"{_BASE}/artifacts/home_credit")
RESULTS_DIR    = ARTIFACTS_DIR / "results"
MODELS_DIR     = ARTIFACTS_DIR / "models"
FIGURES_DIR    = ARTIFACTS_DIR / "figures"
for _d in (ARTIFACTS_DIR, RESULTS_DIR, MODELS_DIR, FIGURES_DIR):
    _d.mkdir(parents=True, exist_ok=True)

print(f"DATA_PATH      = {DATA_PATH}")
print(f"ARTIFACTS_DIR  = {ARTIFACTS_DIR}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# DEV_MODE — smoke-test switch.
#
# When False (the default), every constant resolves to the exact value used in
# the original experiment. The DEV_MODE branches below are dead code in that
# path, so the run is behaviourally identical to the source notebook.
#
# When True, the notebook runs a tiny fast version: a subsample of the data,
# a couple of epochs, a single Optuna trial. Use this on Colab to validate the
# full pipeline end-to-end before launching a real run.
# ──────────────────────────────────────────────────────────────────────────────
DEV_MODE = False
if DEV_MODE:
    print("DEV_MODE is ON — smoke-test run with reduced data/epochs/trials.")

## 2. Data Loading

In [ ]:
# Load preprocessed Home Credit data + drop the leakage / low-signal columns
# identified during initial EDA.
df = pd.read_parquet(DATA_PATH)

DROP_COLUMNS_HC = [
    "FLAG_PHONE", "OCCUPATION_TYPE", "SELLERPLACE_AREA", "AMT_CREDIT_LIMIT_ACTUAL",
    "AMT_DRAWINGS_ATM_CURRENT", "CODE_REJECT_REASON_HC", "AMT_PAYMENT_TOTAL_CURRENT",
    "CHANNEL_TYPE_AP+ (Cash loan)", "NUM_INSTALMENT_NUMBER_x", "CNT_INSTALMENT_MATURE_CUM",
    "CREDIT_TYPE_Car loan", "WEEKDAY_APPR_PROCESS_START_MONDAY",
    "CHANNEL_TYPE_Credit and cash offices", "NAME_YIELD_GROUP_middle",
    "AMT_DRAWINGS_CURRENT", "SK_ID_PREV_y", "CHANNEL_TYPE_Channel of corporate sales",
    "SK_DPD_y", "WEEKDAY_APPR_PROCESS_START_WEDNESDAY", "MONTHS_MAX",
    "NAME_GOODS_CATEGORY_Clothing and Accessories", "CODE_REJECT_REASON_SCO",
    "OBS_30_CNT_SOCIAL_CIRCLE", "AMT_REQ_CREDIT_BUREAU_YEAR",
    "PRODUCT_COMBINATION_Cash Street: middle",
    "NAME_GOODS_CATEGORY_Photo / Cinema Equipment",
    "PRODUCT_COMBINATION_Cash X-Sell: middle", "AMT_APPLICATION", "DAYS_FIRST_DUE",
    "FLAG_DOCUMENT_16", "AMT_ANNUITY", "NAME_SELLER_INDUSTRY_Connectivity",
    "PRODUCT_COMBINATION_POS industry without interest", "NONLIVINGAPARTMENTS_AVG",
    "NAME_SELLER_INDUSTRY_Industry", "DAYS_CREDIT_UPDATE", "RATE_DOWN_PAYMENT",
    "ELEVATORS_AVG", "STATUS_X", "ENTRANCES_AVG",
    "WEEKDAY_APPR_PROCESS_START_SATURDAY", "MONTHS_BALANCE_y",
    "AMT_CREDIT_SUM_OVERDUE", "NAME_PORTFOLIO_POS", "SK_DPD_x", "FLAG_DOCUMENT_13",
    "NAME_TYPE_SUITE_Unaccompanied", "WEEKDAY_APPR_PROCESS_START_FRIDAY",
    "FLAG_DOCUMENT_6", "CHANNEL_TYPE_Stone", "SK_DPD_DEF_y",
    "NAME_PRODUCT_TYPE_x-sell", "NONLIVINGAREA_AVG", "FLAG_DOCUMENT_18",
    "nb_app", "NAME_HOUSING_TYPE", "NAME_PORTFOLIO_Cash",
    "WEEKDAY_APPR_PROCESS_START_SUNDAY", "WEEKDAY_APPR_PROCESS_START_THURSDAY",
    "NFLAG_INSURED_ON_APPROVAL", "MONTHS_COUNT", "YEARS_BUILD_MEDI",
    "DAYS_FIRST_DRAWING", "CODE_REJECT_REASON_LIMIT", "NAME_YIELD_GROUP_XNA",
    "NAME_GOODS_CATEGORY_Medical Supplies", "CREDIT_TYPE_Consumer credit",
    "PRODUCT_COMBINATION_POS household with interest",
    "PRODUCT_COMBINATION_POS mobile with interest", "AMT_DRAWINGS_POS_CURRENT",
    "NAME_CONTRACT_TYPE_Revolving loans", "FLOORSMAX_AVG", "WALLSMATERIAL_MODE",
    "CHANNEL_TYPE_Contact center", "CREDIT_TYPE_Credit card",
    "CODE_REJECT_REASON_SCOFR", "NAME_SELLER_INDUSTRY_XNA", "BASEMENTAREA_AVG",
    "LANDAREA_MEDI", "NAME_SELLER_INDUSTRY_Consumer electronics", "NAME_TYPE_SUITE",
    "FLAG_LAST_APPL_PER_CONTRACT_Y", "CNT_CHILDREN",
    "PRODUCT_COMBINATION_Cash Street: high", "NAME_GOODS_CATEGORY_Sport and Leisure",
    "NAME_TYPE_SUITE_Children", "NAME_SELLER_INDUSTRY_Construction",
    "NAME_GOODS_CATEGORY_Tourism", "NAME_CASH_LOAN_PURPOSE_Repairs",
    "FLAG_CONT_MOBILE", "WEEKDAY_APPR_PROCESS_START_TUESDAY",
    "AMT_REQ_CREDIT_BUREAU_DAY", "NAME_GOODS_CATEGORY_Mobile",
    "NAME_TYPE_SUITE_Spouse, partner", "REG_REGION_NOT_LIVE_REGION",
    "CREDIT_TYPE_Another type of loan", "COMMONAREA_MEDI", "FONDKAPREMONT_MODE",
    "NAME_GOODS_CATEGORY_Construction Materials", "NAME_PORTFOLIO_XNA",
    "NAME_TYPE_SUITE_Family", "NAME_PAYMENT_TYPE_Non-cash from your account",
    "NAME_CASH_LOAN_PURPOSE_Other", "CNT_FAM_MEMBERS", "CREDIT_ACTIVE_Sold",
    "PRODUCT_COMBINATION_Card Street", "PRODUCT_COMBINATION_Cash",
    "NAME_TYPE_SUITE_Other_B", "NAME_CONTRACT_TYPE_Cash loans", "FLOORSMIN_AVG",
    "CHANNEL_TYPE_Country-wide", "NUM_INSTALMENT_VERSION",
    "NAME_CONTRACT_STATUS_Canceled", "AMT_REQ_CREDIT_BUREAU_MON",
    "NAME_TYPE_SUITE_Other_A", "NAME_SELLER_INDUSTRY_MLM partners",
    "NAME_GOODS_CATEGORY_Gardening", "REG_CITY_NOT_WORK_CITY",
    "CNT_CREDIT_PROLONG", "NUM_INSTALMENT_NUMBER",
    "NAME_GOODS_CATEGORY_Auto Accessories", "STATUS_C", "NAME_GOODS_CATEGORY_Jewelry",
    "AMT_REQ_CREDIT_BUREAU_WEEK", "CHANNEL_TYPE_Car dealer", "HOUSETYPE_MODE",
    "FLAG_MOBIL", "FLAG_EMAIL", "FLAG_EMP_PHONE",
    "LIVE_REGION_NOT_WORK_REGION", "LIVE_CITY_NOT_WORK_CITY",
    "NAME_GOODS_CATEGORY_Fitness", "NAME_GOODS_CATEGORY_Education",
    "NAME_GOODS_CATEGORY_Direct Sales", "NAME_GOODS_CATEGORY_Consumer Electronics",
    "NAME_GOODS_CATEGORY_Computers", "NAME_GOODS_CATEGORY_Audio/Video",
    "NAME_GOODS_CATEGORY_Additional Service", "NAME_GOODS_CATEGORY_Animals",
    "NAME_TYPE_SUITE_Group of people", "NAME_CLIENT_TYPE_Refreshed",
    "NAME_CLIENT_TYPE_XNA", "AMT_DRAWINGS_OTHER_CURRENT",
    "CREDIT_TYPE_Unknown type of loan", "CODE_REJECT_REASON_SYSTEM",
    "CODE_REJECT_REASON_XNA", "CODE_REJECT_REASON_VERIF",
    "CREDIT_TYPE_Real estate loan", "NUNIQUE_STATUS_x", "NUNIQUE_STATUS_y",
    "NUNIQUE_STATUS2_y", "CNT_DRAWINGS_OTHER_CURRENT",
    "REG_REGION_NOT_WORK_REGION", "NAME_GOODS_CATEGORY_House Construction",
    "NAME_GOODS_CATEGORY_Homewares", "NAME_CASH_LOAN_PURPOSE_Buying a new car",
    "NAME_CASH_LOAN_PURPOSE_Buying a used car", "NAME_CASH_LOAN_PURPOSE_Car repairs",
    "NAME_CASH_LOAN_PURPOSE_Education",
    "NAME_CASH_LOAN_PURPOSE_Business development",
    "NAME_CASH_LOAN_PURPOSE_Buying a garage", "FLAG_LAST_APPL_PER_CONTRACT_N",
    "NAME_CASH_LOAN_PURPOSE_Building a house or an annex",
    "NFLAG_LAST_APPL_IN_DAY", "CHANNEL_TYPE_Regional / Local",
    "NAME_SELLER_INDUSTRY_Auto technology", "FLAG_DOCUMENT_17",
    "FLAG_DOCUMENT_19", "FLAG_DOCUMENT_20", "FLAG_DOCUMENT_21",
    "AMT_REQ_CREDIT_BUREAU_HOUR", "NAME_GOODS_CATEGORY_Other",
    "NAME_GOODS_CATEGORY_Office Appliances", "NAME_GOODS_CATEGORY_Insurance",
    "NAME_GOODS_CATEGORY_Medicine", "NAME_GOODS_CATEGORY_Weapon",
    "NAME_GOODS_CATEGORY_Vehicles", "NAME_CASH_LOAN_PURPOSE_Buying a holiday home / land",
    "NAME_CASH_LOAN_PURPOSE_Buying a home", "NAME_CASH_LOAN_PURPOSE_Hobby",
    "NAME_CASH_LOAN_PURPOSE_Gasification / water supply",
    "NAME_CASH_LOAN_PURPOSE_Furniture", "NAME_CASH_LOAN_PURPOSE_Everyday expenses",
    "NAME_CASH_LOAN_PURPOSE_Money for a third person",
    "NAME_CASH_LOAN_PURPOSE_Payments on other loans",
    "NAME_CASH_LOAN_PURPOSE_Medicine", "NAME_CASH_LOAN_PURPOSE_Journey",
    "NAME_CASH_LOAN_PURPOSE_Refusal to name the goal",
    "NAME_CASH_LOAN_PURPOSE_Purchase of electronic equipment",
    "NAME_CASH_LOAN_PURPOSE_Wedding / gift / holiday",
    "NAME_CASH_LOAN_PURPOSE_Urgent needs",
    "NAME_CONTRACT_STATUS_Unused offer",
    "NAME_PAYMENT_TYPE_Cashless from the account of the employer",
    "NAME_CONTRACT_TYPE_XNA",
    "PRODUCT_COMBINATION_POS household without interest",
    "FLAG_DOCUMENT_5", "FLAG_DOCUMENT_4", "FLAG_DOCUMENT_2",
    "EMERGENCYSTATE_MODE",
    "PRODUCT_COMBINATION_POS others without interest",
    "PRODUCT_COMBINATION_POS other with interest",
    "PRODUCT_COMBINATION_POS mobile without interest", "CREDIT_DAY_OVERDUE",
    "NAME_SELLER_INDUSTRY_Tourism", "NAME_SELLER_INDUSTRY_Jewelry",
    "PRODUCT_COMBINATION_Card X-Sell", "FLAG_DOCUMENT_15",
    "FLAG_DOCUMENT_7", "FLAG_DOCUMENT_8", "FLAG_DOCUMENT_9",
    "FLAG_DOCUMENT_10", "CREDIT_CURRENCY_currency 4",
    "CREDIT_CURRENCY_currency 3", "CREDIT_CURRENCY_currency 2",
    "CREDIT_ACTIVE_Bad debt", "FLAG_DOCUMENT_11", "FLAG_DOCUMENT_12",
    "FLAG_DOCUMENT_14", "CREDIT_TYPE_Cash loan (non-earmarked)",
    "CREDIT_TYPE_Loan for the purchase of equipment",
    "CREDIT_TYPE_Loan for purchase of shares (margin lending)",
    "CREDIT_TYPE_Loan for business development",
    "CREDIT_TYPE_Interbank credit",
    "CREDIT_TYPE_Loan for working capital replenishment",
    "CREDIT_TYPE_Mobile operator loan", "FLAG_OWN_REALTY", "FLAG_OWN_CAR",
]
df = df.drop(columns=DROP_COLUMNS_HC)

NUM_COLS_HC = [
    "AMT_INCOME_TOTAL", "AMT_CREDIT_x", "AMT_ANNUITY_x", "AMT_GOODS_PRICE_x",
    "REGION_POPULATION_RELATIVE", "DAYS_BIRTH", "DAYS_EMPLOYED",
    "DAYS_REGISTRATION", "DAYS_ID_PUBLISH", "OWN_CAR_AGE",
    "REGION_RATING_CLIENT", "REGION_RATING_CLIENT_W_CITY",
    "HOUR_APPR_PROCESS_START_x", "REG_CITY_NOT_LIVE_CITY",
    "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3",
    "APARTMENTS_AVG", "YEARS_BEGINEXPLUATATION_AVG", "YEARS_BUILD_AVG",
    "COMMONAREA_AVG", "LANDAREA_AVG", "LIVINGAPARTMENTS_AVG", "LIVINGAREA_AVG",
    "APARTMENTS_MODE", "BASEMENTAREA_MODE", "YEARS_BEGINEXPLUATATION_MODE",
    "YEARS_BUILD_MODE", "COMMONAREA_MODE", "ELEVATORS_MODE", "ENTRANCES_MODE",
    "FLOORSMAX_MODE", "FLOORSMIN_MODE", "LANDAREA_MODE", "LIVINGAPARTMENTS_MODE",
    "LIVINGAREA_MODE", "NONLIVINGAPARTMENTS_MODE", "NONLIVINGAREA_MODE",
    "APARTMENTS_MEDI", "BASEMENTAREA_MEDI", "YEARS_BEGINEXPLUATATION_MEDI",
    "ELEVATORS_MEDI", "ENTRANCES_MEDI", "FLOORSMAX_MEDI", "FLOORSMIN_MEDI",
    "LIVINGAPARTMENTS_MEDI", "LIVINGAREA_MEDI", "NONLIVINGAPARTMENTS_MEDI",
    "NONLIVINGAREA_MEDI", "TOTALAREA_MODE", "DEF_30_CNT_SOCIAL_CIRCLE",
    "OBS_60_CNT_SOCIAL_CIRCLE", "DEF_60_CNT_SOCIAL_CIRCLE",
    "DAYS_LAST_PHONE_CHANGE", "AMT_REQ_CREDIT_BUREAU_QRT", "AMT_ANNUITY_y",
    "AMT_CREDIT_y", "AMT_DOWN_PAYMENT", "AMT_GOODS_PRICE_y",
    "HOUR_APPR_PROCESS_START_y", "RATE_INTEREST_PRIMARY",
    "RATE_INTEREST_PRIVILEGED", "DAYS_DECISION", "CNT_PAYMENT",
    "DAYS_LAST_DUE_1ST_VERSION", "DAYS_LAST_DUE", "DAYS_TERMINATION",
    "NAME_CONTRACT_TYPE_Consumer loans", "NAME_CASH_LOAN_PURPOSE_XAP",
    "NAME_CASH_LOAN_PURPOSE_XNA", "NAME_CONTRACT_STATUS_Approved",
    "NAME_CONTRACT_STATUS_Refused", "NAME_PAYMENT_TYPE_Cash through the bank",
    "NAME_PAYMENT_TYPE_XNA", "CODE_REJECT_REASON_CLIENT",
    "CODE_REJECT_REASON_XAP", "NAME_CLIENT_TYPE_New",
    "NAME_CLIENT_TYPE_Repeater", "NAME_GOODS_CATEGORY_Furniture",
    "NAME_GOODS_CATEGORY_XNA", "NAME_PORTFOLIO_Cards", "NAME_PORTFOLIO_Cars",
    "NAME_PRODUCT_TYPE_XNA", "NAME_PRODUCT_TYPE_walk-in",
    "NAME_SELLER_INDUSTRY_Clothing", "NAME_SELLER_INDUSTRY_Furniture",
    "NAME_YIELD_GROUP_high", "NAME_YIELD_GROUP_low_action",
    "NAME_YIELD_GROUP_low_normal", "PRODUCT_COMBINATION_Cash Street: low",
    "PRODUCT_COMBINATION_Cash X-Sell: high", "PRODUCT_COMBINATION_Cash X-Sell: low",
    "PRODUCT_COMBINATION_POS industry with interest", "DAYS_CREDIT",
    "DAYS_CREDIT_ENDDATE", "DAYS_ENDDATE_FACT", "AMT_CREDIT_MAX_OVERDUE",
    "AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT", "AMT_CREDIT_SUM_LIMIT",
    "STATUS_0", "STATUS_1", "STATUS_2", "STATUS_3", "STATUS_4", "STATUS_5",
    "MONTHS_MIN", "CREDIT_ACTIVE_Active", "CREDIT_ACTIVE_Closed",
    "CREDIT_CURRENCY_currency 1", "CREDIT_TYPE_Microloan", "CREDIT_TYPE_Mortgage",
    "buro_count", "MONTHS_BALANCE_x", "CNT_INSTALMENT", "CNT_INSTALMENT_FUTURE",
    "SK_DPD_DEF_x", "NUNIQUE_STATUS2_x", "AMT_BALANCE",
    "AMT_INST_MIN_REGULARITY", "AMT_PAYMENT_CURRENT",
    "AMT_RECEIVABLE_PRINCIPAL", "AMT_RECIVABLE", "AMT_TOTAL_RECEIVABLE",
    "CNT_DRAWINGS_ATM_CURRENT", "CNT_DRAWINGS_CURRENT",
    "CNT_DRAWINGS_POS_CURRENT", "NUM_INSTALMENT_VERSION_x",
    "DAYS_INSTALMENT_x", "DAYS_ENTRY_PAYMENT_x", "AMT_INSTALMENT_x",
    "AMT_PAYMENT_x", "SK_ID_PREV_x", "NUM_INSTALMENT_VERSION_y",
    "NUM_INSTALMENT_NUMBER_y", "DAYS_INSTALMENT_y", "DAYS_ENTRY_PAYMENT_y",
    "AMT_INSTALMENT_y", "AMT_PAYMENT_y", "DAYS_INSTALMENT",
    "DAYS_ENTRY_PAYMENT", "AMT_INSTALMENT", "AMT_PAYMENT",
]
CAT_COLS_HC = [
    "NAME_CONTRACT_TYPE", "CODE_GENDER", "NAME_INCOME_TYPE",
    "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS",
    "WEEKDAY_APPR_PROCESS_START", "ORGANIZATION_TYPE", "EMERGENCYSTATE_MODE",
]
num_cols = NUM_COLS_HC
cat_cols = CAT_COLS_HC

## 3. Preprocessing Pipeline

In [ ]:
def apply_missing_value_filter(X_train, X_valid, X_test, threshold=0.50):
    """Drop columns whose missing rate in X_train exceeds `threshold`."""
    missing_pct = X_train.isnull().mean()
    cols_to_drop = missing_pct[missing_pct > threshold].index.tolist()
    print("--- 2. Missing Value Column Filter ---")
    print(f"Missing percentage threshold: {threshold:.1%}")
    print(f"Columns dropped ({len(cols_to_drop)}): {cols_to_drop if cols_to_drop else 'None'}")

    X_train_clean = X_train.drop(columns=cols_to_drop, errors="ignore")
    X_valid_clean = X_valid.drop(columns=cols_to_drop, errors="ignore")
    X_test_clean  = X_test.drop(columns=cols_to_drop, errors="ignore")

    all_cols = X_train_clean.columns.tolist()
    num_cols_clean = X_train_clean.select_dtypes(include=np.number).columns.tolist()
    cat_cols_clean = [c for c in all_cols if c not in num_cols_clean]
    print(f"Features remaining: {X_train_clean.shape[1]}")
    print("-" * 50)
    return X_train_clean, X_valid_clean, X_test_clean, num_cols_clean, cat_cols_clean


def apply_correlation_drop(X_train, y_train, X_valid, X_test, threshold=0.9):
    """Drop one feature from each numeric pair whose |correlation| > threshold,
    keeping the one with higher absolute correlation to the target."""
    print("--- 3. Correlation Drop (numeric only) ---")
    all_cols = X_train.columns.tolist()
    num_cols_start = X_train.select_dtypes(include=np.number).columns.tolist()
    cat_cols_start = [c for c in all_cols if c not in num_cols_start]
    X_train_num = X_train[num_cols_start]

    print(f"Numeric features before drop: {len(num_cols_start)}. Threshold: {threshold}")
    corr_matrix = X_train_num.corr().abs()
    target_corr = X_train_num.apply(lambda x: x.corr(y_train)).abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

    to_drop_num = set()
    for col in upper.columns:
        for row in upper.index:
            if upper.loc[row, col] > threshold:
                a, b = row, col
                if a in to_drop_num or b in to_drop_num:
                    continue
                if target_corr.get(a, 0) < target_corr.get(b, 0):
                    to_drop_num.add(a)
                else:
                    to_drop_num.add(b)

    features_to_drop = list(to_drop_num)
    X_train_clean = X_train.drop(columns=features_to_drop, errors="ignore")
    X_valid_clean = X_valid.drop(columns=features_to_drop, errors="ignore")
    X_test_clean  = X_test.drop(columns=features_to_drop, errors="ignore")
    print(f"Numeric features dropped: {len(features_to_drop)}")
    print("-" * 50)

    num_cols_clean = [c for c in num_cols_start if c not in features_to_drop]
    cat_cols_clean = cat_cols_start
    return X_train_clean, X_valid_clean, X_test_clean, num_cols_clean, cat_cols_clean


def preprocess_data_pipeline(df, target="TARGET", corr_threshold=0.9, missing_threshold=0.50):
    """Full HC preprocessing pipeline: split, filter missing, drop correlated,
    impute, label-encode, scale. Returns processed splits and helper objects."""
    print("--- 1. Splitting Data (64/16/20) ---")
    X = df.drop(columns=[target])
    y = df[target]
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=0.20, stratify=y, random_state=42,
    )
    X_train, X_valid, y_train, y_valid = train_test_split(
        X_temp, y_temp, test_size=0.20, stratify=y_temp, random_state=42,
    )
    print(f"Split sizes: Train={len(X_train)} | Valid={len(X_valid)} | Test={len(X_test)}")
    print("-" * 50)

    num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
    cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

    X_train, X_valid, X_test, num_cols, cat_cols = apply_missing_value_filter(
        X_train, X_valid, X_test, threshold=missing_threshold,
    )
    X_train, X_valid, X_test, num_cols, cat_cols = apply_correlation_drop(
        X_train, y_train, X_valid, X_test, threshold=corr_threshold,
    )

    print("--- 4. Leakage-Free Transformations ---")
    median_imputers = {}
    if num_cols:
        print(f"Imputing {len(num_cols)} numeric columns with median...")
        for col in num_cols:
            median_value = X_train[col].median()
            median_imputers[col] = median_value
            X_train[col] = X_train[col].fillna(median_value)
            X_valid[col] = X_valid[col].fillna(median_value)
            X_test[col]  = X_test[col].fillna(median_value)

    encoders = {}
    cat_cardinalities = []
    for col in cat_cols:
        X_train[col] = X_train[col].fillna("Missing")
        X_valid[col] = X_valid[col].fillna("Missing")
        X_test[col]  = X_test[col].fillna("Missing")

        le = LabelEncoder()
        le.fit(X_train[col])
        new_classes = list(le.classes_)
        if "Missing" not in new_classes:
            new_classes.append("Missing")
        new_classes.append("UNKNOWN")
        le.classes_ = np.array(new_classes)
        X_train[col] = le.transform(X_train[col])
        encoders[col] = le
        cat_cardinalities.append(len(le.classes_))

        mapping = {cls: i for i, cls in enumerate(le.classes_)}
        unknown_id = mapping["UNKNOWN"]
        X_valid[col] = (X_valid[col].fillna("Missing").astype(str)
                                       .map(mapping).fillna(unknown_id).astype(int))
        X_test[col]  = (X_test[col].fillna("Missing").astype(str)
                                       .map(mapping).fillna(unknown_id).astype(int))

    print(f"Categorical features encoded: {len(cat_cols)}")

    scaler = StandardScaler()
    if num_cols:
        print(f"Scaling {len(num_cols)} numeric columns...")
        X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
        X_valid[num_cols] = scaler.transform(X_valid[num_cols])
        X_test[num_cols]  = scaler.transform(X_test[num_cols])
    print("-" * 50)

    print("--- Final Feature Summary ---")
    print(f"Numeric features      : {len(num_cols)}")
    print(f"Categorical features  : {len(cat_cols)}")
    print(f"Cat cardinalities     : {cat_cardinalities}")
    print(f"Total features        : {X_train.shape[1]}")
    print("-" * 50)

    return (
        X_train, y_train, X_valid, y_valid, X_test, y_test,
        num_cols, cat_cols, encoders, median_imputers, scaler, cat_cardinalities,
    )


def home_credit_summary(df, num_cols, cat_cols, target="TARGET"):
    print("\n" + "=" * 60)
    print("DATASET SUMMARY")
    print("=" * 60)
    print(f"Total Rows: {df.shape[0]:,}")
    print(f"Total Columns: {df.shape[1]:,}\n")
    print("Feature Breakdown")
    print(f"  Numeric Features     : {len(num_cols)}")
    print(f"  Categorical Features : {len(cat_cols)}")
    print(f"  Binary FLAG Columns  : {len([c for c in df.columns if 'FLAG' in c])}")
    print("-" * 50)
    if target in df.columns:
        target_dist = df[target].value_counts(normalize=True) * 100
        print("Target Distribution")
        print(target_dist.to_frame("Percentage %"))
        print("-" * 50)
    print("Missing Value Summary (Top 15)")
    missing = df.isnull().mean().sort_values(ascending=False)
    print((missing * 100).head(15).to_frame("Missing %"))
    print("=" * 60)


home_credit_summary(df, num_cols, cat_cols)
(
    X_train, y_train, X_valid, y_valid, X_test, y_test,
    final_num_cols, final_cat_cols, encoders, imputers, scaler, cat_cardinalities_final,
) = preprocess_data_pipeline(df, target="TARGET", corr_threshold=0.9, missing_threshold=0.9)

print("\n================ Dataset Split Summary ================\n")
print(f"Train : X = {X_train.shape},  y = {y_train.shape}")
print(f"Valid : X = {X_valid.shape},  y = {y_valid.shape}")
print(f"Test  : X = {X_test.shape},   y = {y_test.shape}")
print(f"\nNumeric features      : {len(final_num_cols)}")
print(f"Categorical features  : {len(final_cat_cols)}")
print(f"Cat cardinalities     : {cat_cardinalities_final}")

In [ ]:
# Build torch tensors for FTT-style models.
data_numpy = {"train": {}, "val": {}, "test": {}}
data_numpy["train"]["x_cat"]  = X_train[final_cat_cols]
data_numpy["val"]["x_cat"]    = X_valid[final_cat_cols]
data_numpy["test"]["x_cat"]   = X_test[final_cat_cols]
data_numpy["train"]["X_cont"] = X_train[final_num_cols]
data_numpy["val"]["X_cont"]   = X_valid[final_num_cols]
data_numpy["test"]["X_cont"]  = X_test[final_num_cols]
data_numpy["train"]["y"]      = y_train
data_numpy["val"]["y"]        = y_valid
data_numpy["test"]["y"]       = y_test

data = {}
for part in data_numpy:
    data[part] = {
        "X_cont": torch.tensor(data_numpy[part]["X_cont"].to_numpy(), dtype=torch.float32, device=device),
        "x_cat":  torch.tensor(data_numpy[part]["x_cat"].to_numpy(),  dtype=torch.long,    device=device),
        "y":      torch.tensor(data_numpy[part]["y"].to_numpy(),      dtype=torch.float32, device=device),
    }

d_numerical = len(final_num_cols)
n_train = data["train"]["y"].shape[0]
print(f"d_numerical = {d_numerical}, n_train = {n_train}")

In [ ]:
# When DEV_MODE is on, subsample data to a few thousand rows so a full training
# pass completes in a couple of minutes. No-op when DEV_MODE is False.
if DEV_MODE:
    _n_dev = 5000
    for _part in data:
        _idx = torch.randperm(data[_part]["y"].shape[0], device=device)[:_n_dev]
        for _k in data[_part]:
            data[_part][_k] = data[_part][_k][_idx]
    print({_part: {k: v.shape for k, v in data[_part].items()} for _part in data})

## 4. Shared Training Constants & Helpers

In [ ]:
n_epochs = 100
patience = 10
Lr=2e-4
early_stopping = delu.tools.EarlyStopping(patience, mode="max")
loss_fn = F.binary_cross_entropy_with_logits

def apply_model(batch: Dict[str, Tensor]) -> Tensor:
        return model(batch["X_cont"], batch.get("x_cat")).squeeze(-1)

@torch.no_grad()
def evaluate(part: str) -> float:
    model.eval()
    eval_batch_size = BATCH_SIZE
    y_pred = (
        torch.cat(
            [
                apply_model(batch)
                for batch in delu.iter_batches(data[part], eval_batch_size)
            ]
        )
        .cpu()
        .numpy()
    )
    y_true = data[part]["y"].cpu().numpy()
    y_pred = scipy.special.expit(y_pred)
    auc = roc_auc_score(y_true, y_pred)
    return auc # The higher -- the better.

In [ ]:
import itertools
import typing
import warnings
from collections import OrderedDict
from typing import Any, Dict, Iterable, List, Literal, Optional, Tuple, cast

import torch.optim
from torch.nn.parameter import Parameter

_INTERNAL_ERROR = 'Internal error'

class CategoricalEmbeddings(nn.Module):
    def __init__(
        self, cardinalities: List[int], d_embedding: int, bias: bool = True
    ) -> None:
        super().__init__()
        if not cardinalities:
            raise ValueError('cardinalities must not be empty')
        if any(x <= 0 for x in cardinalities):
            i, value = next((i, x) for i, x in enumerate(cardinalities) if x <= 0)
            raise ValueError(
                'cardinalities must contain only positive values,'
                f' however: cardinalities[{i}]={value}'
            )
        if d_embedding <= 0:
            raise ValueError(f'd_embedding must be positive, however: {d_embedding=}')

        self.embeddings = nn.ModuleList(
            [nn.Embedding(x, d_embedding) for x in cardinalities]
        )
        self.bias = (
            Parameter(torch.empty(len(cardinalities), d_embedding)) if bias else None
        )
        self.reset_parameters()

    def reset_parameters(self) -> None:
        d_rsqrt = self.embeddings[0].embedding_dim ** -0.5
        for m in self.embeddings:
            nn.init.uniform_(m.weight, -d_rsqrt, d_rsqrt)
        if self.bias is not None:
            nn.init.uniform_(self.bias, -d_rsqrt, d_rsqrt)

    def forward(self, x: Tensor) -> Tensor:
        """Do the forward pass."""
        if x.ndim < 2:
            raise ValueError(
                f'The input must have at least two dimensions, however: {x.ndim=}'
            )
        n_features = len(self.embeddings)
        if x.shape[-1] != n_features:
            raise ValueError(
                'The last input dimension (the number of categorical features) must be'
                ' equal to the number of cardinalities passed to the constructor.'
                f' However: {x.shape[-1]=}, len(cardinalities)={n_features}'
            )

        x = torch.stack(
            [self.embeddings[i](x[..., i]) for i in range(n_features)], dim=-2
        )
        if self.bias is not None:
            x = x + self.bias
        return x


_LINFORMER_KV_COMPRESSION_SHARING = Literal['headwise', 'key-value']

class _ReGLU(nn.Module):
    def forward(self, x: Tensor) -> Tensor:
        if x.shape[-1] % 2:
            raise ValueError(
                'For the ReGLU activation, the last input dimension'
                f' must be a multiple of 2, however: {x.shape[-1]=}'
            )
        a, b = x.chunk(2, dim=-1)
        return a * F.relu(b)

class _SwiGLU(nn.Module):
    def __init__(self) -> None:
        """
        Args:
            n_features: the number of continuous features.
            d_embedding: the embedding size.
        """
        super().__init__()
        self.activation = nn.SiLU()

    # def get_output_shape(self) -> torch.Size:
    # # The final output size is (n_features, d_embedding)
    # return torch.Size([self.linear.weight.shape[0], self.d_embedding])

    def forward(self, x: Tensor) -> Tensor:
        """Do the forward pass (SwiGLU = (Main Linear) * Swish(Gate Linear))."""
        A, B = torch.chunk(x, chunks=2, dim=-1)
        A_activated = self.activation(A)
        return A_activated * B


_TransformerFFNActivation = Literal['ReLU', 'ReGLU','SwiGLU']

class MultiheadAttention(nn.Module):
    def __init__(
        self,
        *,
        d_embedding: int,
        n_heads: int,
        dropout: float,
        # Linformer arguments.
        n_tokens: Optional[int] = None,
        linformer_kv_compression_ratio: Optional[float] = None,
        linformer_kv_compression_sharing: Optional[
            _LINFORMER_KV_COMPRESSION_SHARING
        ] = None,
    ) -> None:
        if n_heads < 1:
            raise ValueError(f'n_heads must be positive, however: {n_heads=}')
        if d_embedding % n_heads:
            raise ValueError(
                'd_embedding must be a multiple of n_heads,'
                f' however: {d_embedding=}, {n_heads=}'
            )

        super().__init__()
        self.W_q = nn.Linear(d_embedding, d_embedding)
        self.W_k = nn.Linear(d_embedding, d_embedding)
        self.W_v = nn.Linear(d_embedding, d_embedding)
        self.W_out = nn.Linear(d_embedding, d_embedding) if n_heads > 1 else None
        self.dropout = nn.Dropout(dropout) if dropout else None
        self._n_heads = n_heads

        if linformer_kv_compression_ratio is not None:
            if n_tokens is None:
                raise ValueError(
                    'If linformer_kv_compression_ratio is not None,'
                    ' then n_tokens also must not be None'
                )
            if linformer_kv_compression_sharing not in typing.get_args(
                _LINFORMER_KV_COMPRESSION_SHARING
            ):
                raise ValueError(
                    'Valid values of linformer_kv_compression_sharing include:'
                    f' {typing.get_args(_LINFORMER_KV_COMPRESSION_SHARING)},'
                    f' however: {linformer_kv_compression_sharing=}'
                )
            if (
                linformer_kv_compression_ratio <= 0.0
                or linformer_kv_compression_ratio >= 1.0
            ):
                raise ValueError(
                    'linformer_kv_compression_ratio must be from the open interval'
                    f' (0.0, 1.0), however: {linformer_kv_compression_ratio=}'
                )

            def make_linformer_kv_compression():
                return nn.Linear(
                    n_tokens,
                    max(int(n_tokens * linformer_kv_compression_ratio), 1),
                    bias=False,
                )

            self.key_compression = make_linformer_kv_compression()
            self.value_compression = (
                make_linformer_kv_compression()
                if linformer_kv_compression_sharing == 'headwise'
                else None
            )
        else:
            if n_tokens is not None:
                raise ValueError(
                    'If linformer_kv_compression_ratio is None,'
                    ' then n_tokens also must be None'
                )
            if linformer_kv_compression_sharing is not None:
                raise ValueError(
                    'If linformer_kv_compression_ratio is None,'
                    ' then linformer_kv_compression_sharing also must be None'
                )
            self.key_compression = None
            self.value_compression = None

        for m in [self.W_q, self.W_k, self.W_v]:
            nn.init.zeros_(m.bias)
        if self.W_out is not None:
            nn.init.zeros_(self.W_out.bias)

    def _reshape(self, x: Tensor) -> Tensor:
        batch_size, n_tokens, d = x.shape
        d_head = d // self._n_heads
        return (
            x.reshape(batch_size, n_tokens, self._n_heads, d_head)
            .transpose(1, 2)
            .reshape(batch_size * self._n_heads, n_tokens, d_head)
        )

    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        """Do the forward pass."""
        q, k, v = self.W_q(x_q), self.W_k(x_kv), self.W_v(x_kv)
        if self.key_compression is not None:
            k = self.key_compression(k.transpose(1, 2)).transpose(1, 2)
            v = (
                self.key_compression
                if self.value_compression is None
                else self.value_compression
            )(v.transpose(1, 2)).transpose(1, 2)

        batch_size = len(q)
        d_head_key = k.shape[-1] // self._n_heads
        d_head_value = v.shape[-1] // self._n_heads
        n_q_tokens = q.shape[1]

        q = self._reshape(q)
        k = self._reshape(k)
        attention_logits = q @ k.transpose(1, 2) / math.sqrt(d_head_key)
        attention_probs = F.softmax(attention_logits, dim=-1)
        if self.dropout is not None:
            attention_probs = self.dropout(attention_probs)
        x = attention_probs @ self._reshape(v)
        x = (
            x.reshape(batch_size, self._n_heads, n_q_tokens, d_head_value)
            .transpose(1, 2)
            .reshape(batch_size, n_q_tokens, self._n_heads * d_head_value)
        )
        if self.W_out is not None:
            x = self.W_out(x)
        return x

class _CLSEmbedding(nn.Module):
    def __init__(self, d_embedding: int) -> None:
        super().__init__()
        self.weight = Parameter(torch.empty(d_embedding))
        self.reset_parameters()

    def reset_parameters(self) -> None:
        d_rsqrt = self.weight.shape[-1] ** -0.5
        nn.init.uniform_(self.weight, -d_rsqrt, d_rsqrt)

    def forward(self, batch_dims: Tuple[int]) -> Tensor:
        if not batch_dims:
            raise ValueError('The input must be non-empty')

        return self.weight.expand(*batch_dims, 1, -1)

class FTTransformerBackbone(nn.Module):
    """The backbone of FT-Transformer.

    The differences with Transformer from the paper
    ["Attention Is All You Need"](https://arxiv.org/abs/1706.03762) are as follows:

    - the so called "PreNorm" variation is used
        (`norm_first=True` in terms of `torch.nn.TransformerEncoderLayer`)
    - the very first normalization is skipped. This is **CRUCIAL** for FT-Transformer
        in the PreNorm configuration.
    """

    def __init__(
        self,
        *,
        d_out: Optional[int],
        n_blocks: int,
        d_block: int,
        attention_n_heads: int,
        attention_dropout: float,
        ffn_d_hidden: Optional[int] = None,
        ffn_d_hidden_multiplier: Optional[float],
        ffn_dropout: float,
        # NOTE[DIFF]
        # In the paper, FT-Transformer uses the ReGLU activation.
        # Here, to illustrate the difference, ReLU activation is also supported
        # (in particular, see the docstring).
        ffn_activation: _TransformerFFNActivation = 'ReGLU',
        residual_dropout: float,
        n_tokens: Optional[int] = None,
        linformer_kv_compression_ratio: Optional[float] = None,
        linformer_kv_compression_sharing: Optional[
            _LINFORMER_KV_COMPRESSION_SHARING
        ] = None,
    ):
        # noqa: E501
        if ffn_activation not in typing.get_args(_TransformerFFNActivation):
            raise ValueError(
                'ffn_activation must be one of'
                f' {typing.get_args(_TransformerFFNActivation)}.'
                f' However: {ffn_activation=}'
            )
        if ffn_d_hidden is None:
            if ffn_d_hidden_multiplier is None:
                raise ValueError(
                    'If ffn_d_hidden is None,'
                    ' then ffn_d_hidden_multiplier must not be None'
                )
            ffn_d_hidden = int(d_block * cast(float, ffn_d_hidden_multiplier))
        else:
            if ffn_d_hidden_multiplier is not None:
                raise ValueError(
                    'If ffn_d_hidden is not None,'
                    ' then ffn_d_hidden_multiplier must be None'
                )

        super().__init__()
        ffn_use_reglu = (ffn_activation == 'ReGLU') or (ffn_activation == 'SwiGLU')
        ffn_use_swiglu = ffn_activation == 'SwiGLU'
        self.blocks = nn.ModuleList(
            [
                nn.ModuleDict(
                    {
                        # >>> attention
                        'attention': MultiheadAttention(
                            d_embedding=d_block,
                            n_heads=attention_n_heads,
                            dropout=attention_dropout,
                            n_tokens=n_tokens,
                            linformer_kv_compression_ratio=linformer_kv_compression_ratio,
                            linformer_kv_compression_sharing=linformer_kv_compression_sharing,
                        ),
                        'attention_residual_dropout': nn.Dropout(residual_dropout),
                        # >>> feed-forward
                        'ffn_normalization': nn.LayerNorm(d_block),
                        'ffn': _named_sequential(
                            (
                                'linear1',
                                # ReGLU divides dimension by 2,
                                # so multiplying by 2 to compensate for this.
                                nn.Linear(
                                    d_block, ffn_d_hidden * (2 if ffn_use_reglu else 1)
                                ),
                            ),
                            ('activation', _SwiGLU() if ffn_use_swiglu else _ReGLU() if ffn_use_reglu else nn.ReLU()),
                            ('dropout', nn.Dropout(ffn_dropout)),
                            ('linear2', nn.Linear(ffn_d_hidden, d_block)),
                        ),
                        'ffn_residual_dropout': nn.Dropout(residual_dropout),
                        # >>> output (for hook-based introspection)
                        'output': nn.Identity(),
                        # >>> the very first normalization
                        **(
                            {}
                            if layer_idx == 0
                            else {'attention_normalization': nn.LayerNorm(d_block)}
                        ),
                    }
                )
                for layer_idx in range(n_blocks)
            ]
        )
        self.output = (
            None
            if d_out is None
            else _named_sequential(
                ('normalization', nn.LayerNorm(d_block)),
                ('activation', nn.ReLU()),
                ('linear', nn.Linear(d_block, d_out)),
            )
        )

    def forward(self, x: Tensor) -> Tensor:
        """Do the forward pass."""
        if x.ndim != 3:
            raise ValueError(
                f'The input must have exactly three dimension, however: {x.ndim=}'
            )

        n_blocks = len(self.blocks)
        for i_block, block in enumerate(self.blocks):
            block = cast(nn.ModuleDict, block)

            x_identity = x
            if 'attention_normalization' in block:
                x = block['attention_normalization'](x)
            x = block['attention'](x[:, :1] if i_block + 1 == n_blocks else x, x)
            x = block['attention_residual_dropout'](x)
            x = x_identity + x

            x_identity = x
            x = block['ffn_normalization'](x)
            x = block['ffn'](x)
            x = block['ffn_residual_dropout'](x)
            x = x_identity + x

            x = block['output'](x)

        x = x[:, 0] # The representation of [CLS]-token.

        if self.output is not None:
            x = self.output(x)
        return x

In [ ]:
def _named_sequential(*modules) -> nn.Sequential:
    return nn.Sequential(OrderedDict(modules))


def _check_input_shape(x: Tensor, expected_n_features: int) -> None:
    if x.ndim < 1:
        raise ValueError(
            f'The input must have at least one dimension, however: {x.ndim=}'
        )
    if x.shape[-1] != expected_n_features:
        raise ValueError(
            'The last dimension of the input was expected to be'
            f' {expected_n_features}, however, {x.shape[-1]=}'
        )


class LinearEmbeddings(nn.Module):
    """Linear embeddings for continuous features.
    **Shape**
    - Input: `(*, n_features)`
    - Output: `(*, n_features, d_embedding)`
    """

    def __init__(self, n_features: int, d_embedding: int) -> None:
        """
        Args:
            n_features: the number of continuous features.
            d_embedding: the embedding size.
        """
        if n_features <= 0:
            raise ValueError(f'n_features must be positive, however: {n_features=}')
        if d_embedding <= 0:
            raise ValueError(f'd_embedding must be positive, however: {d_embedding=}')

        super().__init__()
        self.weight = Parameter(torch.empty(n_features, d_embedding))
        self.bias = Parameter(torch.empty(n_features, d_embedding))
        self.reset_parameters()

    def reset_parameters(self) -> None:
        d_rqsrt = self.weight.shape[1] ** -0.5
        nn.init.uniform_(self.weight, -d_rqsrt, d_rqsrt)
        nn.init.uniform_(self.bias, -d_rqsrt, d_rqsrt)

    def get_output_shape(self) -> torch.Size:
        """Get the output shape without the batch dimensions."""
        return self.weight.shape

    def forward(self, x: Tensor) -> Tensor:
        """Do the forward pass."""
        _check_input_shape(x, self.weight.shape[0])
        return torch.addcmul(self.bias, self.weight, x[..., None])


class LinearSiLUEmbeddings(nn.Module):
    """Simple non-linear embeddings for continuous features.
    .cont_embeddings = (
            LinearSiLUEmbeddings(n_cont_features, d_block)
    **Shape**

    - Input: `(*, n_features)`
    - Output: `(*, n_features, d_embedding)`
    """

    def __init__(self, n_features: int, d_embedding: int = 32) -> None:
        """
        Args:
            n_features: the number of continuous features.
            d_embedding: the embedding size.
        """
        super().__init__()
        # 1. Initialize a single LinearEmbeddings with 2x the output size
        self.d_embedding = d_embedding
        self.linear = LinearEmbeddings(n_features, 2 * d_embedding)
        self.activation = nn.SiLU()

    def get_output_shape(self) -> torch.Size:
        # The final output size is (n_features, d_embedding)
        return torch.Size([self.linear.weight.shape[0], self.d_embedding])

    def forward(self, x: Tensor) -> Tensor:
        """Do the forward pass (SwiGLU = (Main Linear) * Swish(Gate Linear))."""
        x_combined = self.linear(x)
        A, B = torch.chunk(x_combined, chunks=2, dim=-1)
        A_activated = self.activation(A)
        return A_activated * B

## 5. Variant 1 — SwiGLU in Numerical Embeddings

Replaces the default linear-ReLU numerical embedding with a `LinearSiLUEmbeddings` block where the output is gated by a SiLU-activated half (SwiGLU-style). The transformer backbone is otherwise unchanged.

### 5.1 Model

In [ ]:
class FTTransformer(nn.Module):
    def __init__(
        self,
        *,
        n_cont_features: int,
        cat_cardinalities: List[int],
        _is_default: bool = False,
        **backbone_kwargs,
    ) -> None:
        if n_cont_features < 0:
            raise ValueError(
                f'n_cont_features must be non-negative, however: {n_cont_features=}'
            )
        if n_cont_features == 0 and not cat_cardinalities:
            raise ValueError(
                'At least one type of features must be presented, however:'
                f' {n_cont_features=}, {cat_cardinalities=}'
            )
        if 'n_tokens' in backbone_kwargs:
            raise ValueError(
                'backbone_kwargs must not contain key "n_tokens"'
                ' (the number of tokens will be inferred automatically)'
            )

        super().__init__()
        d_block: int = backbone_kwargs['d_block']
        self.cls_embedding = _CLSEmbedding(d_block)

        # >>> Feature embeddings (Figure 2a in the paper).
        self.cont_embeddings = (
            LinearSiLUEmbeddings(n_cont_features, d_block) if n_cont_features > 0 else None
        )
        # self.cont_embeddings = (
        # LinearEmbeddings(n_cont_features, d_block) if n_cont_features > 0 else None
        # )
        self.cat_embeddings = (
            CategoricalEmbeddings(cat_cardinalities, d_block, True)
            if cat_cardinalities
            else None
        )
        # <<<

        self.backbone = FTTransformerBackbone(
            **backbone_kwargs,
            n_tokens=(
                None
                if backbone_kwargs.get('linformer_kv_compression_ratio') is None
                else 1 + n_cont_features + len(cat_cardinalities)
            ),
        )
        self._is_default = _is_default

    @classmethod
    def get_default_kwargs(cls, n_blocks: int = 3) -> Dict[str, Any]:
        """Get the default hyperparameters.

        Args:
            n_blocks: the number of blocks. The supported values are: 1, 2, 3, 4, 5, 6.
        Returns:
            the default keyword arguments for the constructor.
        """
        if n_blocks < 0 or n_blocks > 6:
            raise ValueError(
                'Default configurations are available'
                ' only for the following values of n_blocks: 1, 2, 3, 4, 5, 6.'
                f' However, {n_blocks=}'
            )
        return {
            'n_blocks': n_blocks,
            'd_block': [96, 128, 192, 256, 320, 384][n_blocks - 1],
            'attention_n_heads': 8,
            'attention_dropout': [0.1, 0.15, 0.2, 0.25, 0.3, 0.35][n_blocks - 1],
            'ffn_d_hidden': None,
            # "4 / 3" for ReGLU leads to almost the same number of parameters
            # as "2.0" for ReLU.
            'ffn_d_hidden_multiplier': 4 / 3,
            'ffn_dropout': [0.0, 0.05, 0.1, 0.15, 0.2, 0.25][n_blocks - 1],
            'residual_dropout': 0.0,
            '_is_default': True,
        }

    def make_parameter_groups(self) -> List[Dict[str, Any]]:
        """Make parameter groups for optimizers.

        The difference with calling this method instead of
        `.parameters()` is that this method always sets `weight_decay=0.0`
        for some of the parameters.

        Returns:
            the parameter groups that can be passed to PyTorch optimizers.
        """

        def get_parameters(m: Optional[nn.Module]) -> Iterable[Parameter]:
            return () if m is None else m.parameters()

        zero_wd_group: Dict[str, Any] = {
            'params': set(
                itertools.chain(
                    get_parameters(self.cls_embedding),
                    get_parameters(self.cont_embeddings),
                    get_parameters(self.cat_embeddings),
                    itertools.chain.from_iterable(
                        m.parameters()
                        for block in self.backbone.blocks
                        for name, m in block.named_children()
                        if name.endswith('_normalization')
                    ),
                    (
                        p
                        for name, p in self.named_parameters()
                        if name.endswith('.bias')
                    ),
                )
            ),
            'weight_decay': 0.0,
        }
        main_group: Dict[str, Any] = {
            'params': [p for p in self.parameters() if p not in zero_wd_group['params']]
        }
        zero_wd_group['params'] = list(zero_wd_group['params'])
        return [main_group, zero_wd_group]

    def make_default_optimizer(self) -> torch.optim.AdamW:
        """Create the "default" `torch.nn.AdamW` suitable for the *default* FT-Transformer.

        Returns:
            the optimizer.
        """ # noqa: E501
        if not self._is_default:
            warnings.warn(
                'The default opimizer is supposed to be used in a combination'
                ' with the default FT-Transformer.'
            )
        return torch.optim.AdamW(
            self.make_parameter_groups(), lr=1e-4, weight_decay=1e-5
        )

    _FORWARD_BAD_ARGS_MESSAGE = (
        'Based on the arguments passed to the constructor of FTTransformer, {}'
    )

    def forward(self, x_cont: Optional[Tensor], x_cat: Optional[Tensor]) -> Tensor:
        """Do the forward pass."""
        x_any = x_cat if x_cont is None else x_cont
        if x_any is None:
            raise ValueError('At least one of x_cont and x_cat must be provided.')

        x_embeddings: List[Tensor] = []
        if self.cls_embedding is not None:
            x_embeddings.append(self.cls_embedding(x_any.shape[:-1]))

        for argname, argvalue, module in [
            ('x_cont', x_cont, self.cont_embeddings),
            ('x_cat', x_cat, self.cat_embeddings),
        ]:
            if module is None:
                if argvalue is not None:
                    raise ValueError(
                        FTTransformer._FORWARD_BAD_ARGS_MESSAGE.format(
                            f'{argname} must be None'
                        )
                    )
            else:
                if argvalue is None:
                    raise ValueError(
                        FTTransformer._FORWARD_BAD_ARGS_MESSAGE.format(
                            f'{argname} must not be None'
                        )
                    )
                x_embeddings.append(module(argvalue))
        assert x_embeddings, _INTERNAL_ERROR
        x = torch.cat(x_embeddings, dim=1)
        x = self.backbone(x)
        return x

### 5.2 Train + Evaluate

In [ ]:
trial_results = []
best_auc_global=0.774
global_model_path = str(MODELS_DIR / "ftt_swiglu_embed_hc.pth")
d_out=1

d_Block = 64
n_Blocks = 3
Attention_n_heads = 4
FFN_d_hidden = 2
Lr=2e-4
BATCH_SIZE=2048
# Print trial parameters at the start of each trial
print("\n" + "="*70)
print(" Hyperparameters:")
print(f" d_token = {d_Block}")
print(f" n_blocks = {n_Blocks}")
print(f" attention_n_heads = {Attention_n_heads}")
print(f" ffn_d_hidden = {FFN_d_hidden}")
print("="*70 + "\n")
model = FTTransformer(
    n_cont_features=d_numerical,
    cat_cardinalities=cat_cardinalities_final,
    n_blocks=n_Blocks,
    d_block=d_Block,
    attention_n_heads=Attention_n_heads,
    ffn_d_hidden_multiplier=FFN_d_hidden,
    attention_dropout=0.05,
    ffn_dropout=0.05,
    residual_dropout=0.05,
    d_out=1 # output a single logit (binary)
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=Lr, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
optimizer,
max_lr=Lr * 5, # Recommended: 3–10× base LR
steps_per_epoch=math.ceil(X_train.shape[0] / BATCH_SIZE), # NUMBER OF BATCHES PER EPOCH
epochs=n_epochs,
pct_start=0.2,
anneal_strategy="cos"
)
batch_size = BATCH_SIZE
epoch_size = math.ceil(X_train.shape[0] / batch_size)
timer = delu.tools.Timer()


n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f" Trainable params: {n_params:,}")

print(f'Test score before training: AUC Val = {evaluate("val"):.4f}')


best = {
    "val": -math.inf,
    "test": -math.inf,
    "epoch": -1,
}

print(f"Device: {device.type.upper()}")
print("-" * 88 + "\n")
timer.run()
for epoch in range(n_epochs):
    pos_weight = torch.tensor(6, device=device)
    neg_weight = torch.tensor(1, device=device)
    for batch in tqdm(
        delu.iter_batches(data["train"], batch_size, shuffle=True),
        desc=f"Epoch {epoch}",
        total=epoch_size,
    ):
        model.train()
        optimizer.zero_grad()

        y_batch = batch["y"].float()
        weights = torch.where(
            y_batch == 1,
            pos_weight,
            neg_weight
        )

        loss = loss_fn(apply_model(batch), batch["y"].float(), weight=weights)
        loss.backward()
        optimizer.step()
        scheduler.step()

    val_auc = evaluate("val")
    test_auc = evaluate("test")
    train_auc = evaluate('train')
    print(f" AUC Train {train_auc:.4f} Val {val_auc:.4f} (test) {test_auc:.4f} [time] {timer}")

    early_stopping.update(val_auc)
    if early_stopping.should_stop():
        break

    if val_auc > best["val"]:
        print(" New best epoch! ")
        best = {"val": val_auc, "test": test_auc, "epoch": epoch}
        if val_auc>best_auc_global:
            torch.save(model.state_dict(), global_model_path)
            best_auc_global=val_auc
    print()

trial_results.append({
    'type':'SwiGLU in Numerical Embeddings',
    "auc_val": val_auc,
    "auc_test": test_auc,
    "d_token": d_Block,
    "n_blocks": n_Blocks,
    "n_heads": Attention_n_heads,
    "ffn_d_hidden": FFN_d_hidden,
    'best':best,
    'timer':timer.__str__(),
    'train_auc':train_auc
  })

with open(str(RESULTS_DIR / "ftt_swiglu_embed_hc.json"), "w") as f:
    json.dump(trial_results, f, indent=4)

## 6. Variant 2 — SwiGLU in Attention + FFN

Keeps the embedding layer standard but uses SwiGLU as both the feed-forward activation and as an additional gate after attention.

### 6.1 Model

In [ ]:
class FTTransformer(nn.Module):
    def __init__(
        self,
        *,
        n_cont_features: int,
        cat_cardinalities: List[int],
        _is_default: bool = False,
        **backbone_kwargs,
    ) -> None:
        if n_cont_features < 0:
            raise ValueError(
                f'n_cont_features must be non-negative, however: {n_cont_features=}'
            )
        if n_cont_features == 0 and not cat_cardinalities:
            raise ValueError(
                'At least one type of features must be presented, however:'
                f' {n_cont_features=}, {cat_cardinalities=}'
            )
        if 'n_tokens' in backbone_kwargs:
            raise ValueError(
                'backbone_kwargs must not contain key "n_tokens"'
                ' (the number of tokens will be inferred automatically)'
            )

        super().__init__()
        d_block: int = backbone_kwargs['d_block']
        self.cls_embedding = _CLSEmbedding(d_block)

        # >>> Feature embeddings (Figure 2a in the paper).
        # self.cont_embeddings = (
        # LinearSiLUEmbeddings(n_cont_features, d_block) if n_cont_features > 0 else None
        # )
        self.cont_embeddings = (
            LinearEmbeddings(n_cont_features, d_block) if n_cont_features > 0 else None
        )
        self.cat_embeddings = (
            CategoricalEmbeddings(cat_cardinalities, d_block, True)
            if cat_cardinalities
            else None
        )
        # <<<

        self.backbone = FTTransformerBackbone(
            **backbone_kwargs,
            n_tokens=(
                None
                if backbone_kwargs.get('linformer_kv_compression_ratio') is None
                else 1 + n_cont_features + len(cat_cardinalities)
            ),
        )
        self._is_default = _is_default

    @classmethod
    def get_default_kwargs(cls, n_blocks: int = 3) -> Dict[str, Any]:
        """Get the default hyperparameters.

        Args:
            n_blocks: the number of blocks. The supported values are: 1, 2, 3, 4, 5, 6.
        Returns:
            the default keyword arguments for the constructor.
        """
        if n_blocks < 0 or n_blocks > 6:
            raise ValueError(
                'Default configurations are available'
                ' only for the following values of n_blocks: 1, 2, 3, 4, 5, 6.'
                f' However, {n_blocks=}'
            )
        return {
            'n_blocks': n_blocks,
            'd_block': [96, 128, 192, 256, 320, 384][n_blocks - 1],
            'attention_n_heads': 8,
            'attention_dropout': [0.1, 0.15, 0.2, 0.25, 0.3, 0.35][n_blocks - 1],
            'ffn_d_hidden': None,
            # "4 / 3" for ReGLU leads to almost the same number of parameters
            # as "2.0" for ReLU.
            'ffn_d_hidden_multiplier': 4 / 3,
            'ffn_dropout': [0.0, 0.05, 0.1, 0.15, 0.2, 0.25][n_blocks - 1],
            'residual_dropout': 0.0,
            '_is_default': True,
        }

    def make_parameter_groups(self) -> List[Dict[str, Any]]:
        """Make parameter groups for optimizers.

        The difference with calling this method instead of
        `.parameters()` is that this method always sets `weight_decay=0.0`
        for some of the parameters.

        Returns:
            the parameter groups that can be passed to PyTorch optimizers.
        """

        def get_parameters(m: Optional[nn.Module]) -> Iterable[Parameter]:
            return () if m is None else m.parameters()

        zero_wd_group: Dict[str, Any] = {
            'params': set(
                itertools.chain(
                    get_parameters(self.cls_embedding),
                    get_parameters(self.cont_embeddings),
                    get_parameters(self.cat_embeddings),
                    itertools.chain.from_iterable(
                        m.parameters()
                        for block in self.backbone.blocks
                        for name, m in block.named_children()
                        if name.endswith('_normalization')
                    ),
                    (
                        p
                        for name, p in self.named_parameters()
                        if name.endswith('.bias')
                    ),
                )
            ),
            'weight_decay': 0.0,
        }
        main_group: Dict[str, Any] = {
            'params': [p for p in self.parameters() if p not in zero_wd_group['params']]
        }
        zero_wd_group['params'] = list(zero_wd_group['params'])
        return [main_group, zero_wd_group]

    def make_default_optimizer(self) -> torch.optim.AdamW:
        """Create the "default" `torch.nn.AdamW` suitable for the *default* FT-Transformer.

        Returns:
            the optimizer.
        """ # noqa: E501
        if not self._is_default:
            warnings.warn(
                'The default opimizer is supposed to be used in a combination'
                ' with the default FT-Transformer.'
            )
        return torch.optim.AdamW(
            self.make_parameter_groups(), lr=1e-4, weight_decay=1e-5
        )

    _FORWARD_BAD_ARGS_MESSAGE = (
        'Based on the arguments passed to the constructor of FTTransformer, {}'
    )

    def forward(self, x_cont: Optional[Tensor], x_cat: Optional[Tensor]) -> Tensor:
        """Do the forward pass."""
        x_any = x_cat if x_cont is None else x_cont
        if x_any is None:
            raise ValueError('At least one of x_cont and x_cat must be provided.')

        x_embeddings: List[Tensor] = []
        if self.cls_embedding is not None:
            x_embeddings.append(self.cls_embedding(x_any.shape[:-1]))

        for argname, argvalue, module in [
            ('x_cont', x_cont, self.cont_embeddings),
            ('x_cat', x_cat, self.cat_embeddings),
        ]:
            if module is None:
                if argvalue is not None:
                    raise ValueError(
                        FTTransformer._FORWARD_BAD_ARGS_MESSAGE.format(
                            f'{argname} must be None'
                        )
                    )
            else:
                if argvalue is None:
                    raise ValueError(
                        FTTransformer._FORWARD_BAD_ARGS_MESSAGE.format(
                            f'{argname} must not be None'
                        )
                    )
                x_embeddings.append(module(argvalue))
        assert x_embeddings, _INTERNAL_ERROR
        x = torch.cat(x_embeddings, dim=1)
        x = self.backbone(x)
        return x

### 6.2 Train + Evaluate

In [ ]:
trial_results = []
best_auc_global=0.774
global_model_path = str(MODELS_DIR / "ftt_swiglu_attn_ffn_hc.pth")
d_out=1
early_stopping = delu.tools.EarlyStopping(patience, mode="max")
d_Block = 64
n_Blocks = 3
Attention_n_heads = 4
FFN_d_hidden = 2
Lr=2e-4
BATCH_SIZE=2048+4096
# Print trial parameters at the start of each trial
print("\n" + "="*70)
print(" Hyperparameters:")
print(f" d_token = {d_Block}")
print(f" n_blocks = {n_Blocks}")
print(f" attention_n_heads = {Attention_n_heads}")
print(f" ffn_d_hidden = {FFN_d_hidden}")
print("="*70 + "\n")
model = FTTransformer(
    n_cont_features=d_numerical,
    cat_cardinalities=cat_cardinalities_final,
    n_blocks=n_Blocks,
    d_block=d_Block,
    attention_n_heads=Attention_n_heads,
    ffn_d_hidden_multiplier=FFN_d_hidden,
    ffn_activation='SwiGLU',
    attention_dropout=0.05,
    ffn_dropout=0.05,
    residual_dropout=0.05,
    d_out=1 # output a single logit (binary)
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=Lr, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
optimizer,
max_lr=Lr * 5, # Recommended: 3–10× base LR
steps_per_epoch=math.ceil(X_train.shape[0] / BATCH_SIZE), # NUMBER OF BATCHES PER EPOCH
epochs=n_epochs,
pct_start=0.2,
anneal_strategy="cos"
)
batch_size = BATCH_SIZE
epoch_size = math.ceil(X_train.shape[0] / batch_size)
timer = delu.tools.Timer()


n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f" Trainable params: {n_params:,}")

print(f'Test score before training: AUC Val = {evaluate("val"):.4f}')


best = {
    "val": -math.inf,
    "test": -math.inf,
    "epoch": -1,
}

print(f"Device: {device.type.upper()}")
print("-" * 88 + "\n")
timer.run()
for epoch in range(n_epochs):
    pos_weight = torch.tensor(6, device=device)
    neg_weight = torch.tensor(1, device=device)
    for batch in tqdm(
        delu.iter_batches(data["train"], batch_size, shuffle=True),
        desc=f"Epoch {epoch}",
        total=epoch_size,
    ):
        model.train()
        optimizer.zero_grad()

        y_batch = batch["y"].float()
        weights = torch.where(
            y_batch == 1,
            pos_weight,
            neg_weight
        )

        loss = loss_fn(apply_model(batch), batch["y"].float(), weight=weights)
        loss.backward()
        optimizer.step()
        scheduler.step()

    val_auc = evaluate("val")
    test_auc = evaluate("test")
    train_auc = evaluate('train')
    print(f" AUC Train {train_auc:.4f} Val {val_auc:.4f} (test) {test_auc:.4f} [time] {timer}")

    early_stopping.update(val_auc)
    if early_stopping.should_stop():
      print('Early stopping triggered')
      break

    if val_auc > best["val"]:
        print(" New best epoch! ")
        best = {"val": val_auc, "test": test_auc, "epoch": epoch}
        if val_auc>best_auc_global:
            torch.save(model.state_dict(), global_model_path)
            best_auc_global=val_auc
    print()

trial_results.append({
    'type':'SwiGLU in Attention FFN',
    "auc_val": val_auc,
    "auc_test": test_auc,
    "d_token": d_Block,
    "n_blocks": n_Blocks,
    "n_heads": Attention_n_heads,
    "ffn_d_hidden": FFN_d_hidden,
    'best':best,
    'timer':timer.__str__(),
    'train_auc':train_auc
  })

with open(str(RESULTS_DIR / "ftt_swiglu_attn_ffn_hc.json"), "w") as f:
    json.dump(trial_results, f, indent=4)

## 7. Variant 3 — Cross-Feature SwiGLU Tokenizer

Breaks the per-feature independence assumption of vanilla FTT: each token is constructed from both its own value and a SwiGLU-mixed context computed across all features. Tests whether early cross-feature interaction helps.

### 7.1 Model

In [ ]:
class CrossFeatureSwiGLUTokenizer(nn.Module):
    """
    Cross-feature SwiGLU tokenizer for FTT numerical embeddings.

    Breaks the per-feature independence assumption of vanilla FTT by:
    1. Computing a cross-feature context via SwiGLU over all features
    2. Creating each token using both its own value AND the cross-feature context

    Drop-in replacement for FTT's cont_embeddings.

    Shape:
        Input: (*, n_features)
        Output: (*, n_features, d_block)
    """
    def __init__(self, n_features: int, d_block: int) -> None:
        super().__init__()
        self.n_features = n_features
        self.d_block = d_block

        # ── Stage 1: Cross-feature context via SwiGLU ──
        # Maps all features → shared context vector (B, n_features)
        self.context_gate = nn.Linear(n_features, n_features)
        self.context_value = nn.Linear(n_features, n_features)

        # ── Stage 2: Per-feature token projection with context ──
        # Each token sees: own scalar (1) + full context (n_features) = n_features + 1
        self.token_gate = nn.Linear(n_features + 1, d_block)
        self.token_value = nn.Linear(n_features + 1, d_block)

        self.silu = nn.SiLU()
        self.norm = nn.LayerNorm(d_block)

        self._init_weights()

    def _init_weights(self) -> None:
        for layer in [
            self.context_gate, self.context_value,
            self.token_gate, self.token_value
        ]:
            nn.init.kaiming_uniform_(layer.weight, a=math.sqrt(5))
            nn.init.zeros_(layer.bias)

    def get_output_shape(self) -> torch.Size:
        return torch.Size([self.n_features, self.d_block])

    def forward(self, x: Tensor) -> Tensor:
        """
        Args:
            x: (*, n_features) — raw standardized numerical features
        Returns:
            (*, n_features, d_block) — token embeddings
        """
        # ── Stage 1: Cross-feature context ──
        # SwiGLU over all features — each position attends to all others
        context = self.silu(self.context_gate(x)) * self.context_value(x)
        # context: (*, n_features)

        # ── Stage 2: Vectorized per-feature token creation ──
        # Expand context for all n_features token positions
        # Each token i gets: [context (n_features) | own scalar x_i (1)]

        # context repeated for each feature position: (*, n_features, n_features)
        context_expanded = context.unsqueeze(-2).expand(
            *context.shape[:-1], self.n_features, self.n_features
        )

        # own scalar for each position: (*, n_features, 1)
        x_self = x.unsqueeze(-1)

        # concatenate along last dim: (*, n_features, n_features + 1)
        xi_with_context = torch.cat([context_expanded, x_self], dim=-1)

        # SwiGLU token projection
        gate = self.silu(self.token_gate(xi_with_context)) # (*, n_features, d_block)
        value = self.token_value(xi_with_context) # (*, n_features, d_block)

        out = gate * value # (*, n_features, d_block)
        return self.norm(out)


class FTTransformer(nn.Module):
    def __init__(
        self,
        *,
        n_cont_features: int,
        cat_cardinalities: List[int],
        _is_default: bool = False,
        **backbone_kwargs,
    ) -> None:
        if n_cont_features < 0:
            raise ValueError(
                f'n_cont_features must be non-negative, however: {n_cont_features=}'
            )
        if n_cont_features == 0 and not cat_cardinalities:
            raise ValueError(
                'At least one type of features must be presented, however:'
                f' {n_cont_features=}, {cat_cardinalities=}'
            )
        if 'n_tokens' in backbone_kwargs:
            raise ValueError(
                'backbone_kwargs must not contain key "n_tokens"'
                ' (the number of tokens will be inferred automatically)'
            )

        super().__init__()
        d_block: int = backbone_kwargs['d_block']
        self.cls_embedding = _CLSEmbedding(d_block)

        # >>> Feature embeddings (Figure 2a in the paper).
        # self.cont_embeddings = (
        # LinearSiLUEmbeddings(n_cont_features, d_block) if n_cont_features > 0 else None
        # )
        # self.cont_embeddings = (
        # LinearEmbeddings(n_cont_features, d_block) if n_cont_features > 0 else None
        # )
        self.cont_embeddings = (
            CrossFeatureSwiGLUTokenizer(n_cont_features, d_block)
        )
        self.cat_embeddings = (
            CategoricalEmbeddings(cat_cardinalities, d_block, True)
            if cat_cardinalities
            else None
        )
        # <<<

        self.backbone = FTTransformerBackbone(
            **backbone_kwargs,
            n_tokens=(
                None
                if backbone_kwargs.get('linformer_kv_compression_ratio') is None
                else 1 + n_cont_features + len(cat_cardinalities)
            ),
        )
        self._is_default = _is_default

    @classmethod
    def get_default_kwargs(cls, n_blocks: int = 3) -> Dict[str, Any]:
        """Get the default hyperparameters.

        Args:
            n_blocks: the number of blocks. The supported values are: 1, 2, 3, 4, 5, 6.
        Returns:
            the default keyword arguments for the constructor.
        """
        if n_blocks < 0 or n_blocks > 6:
            raise ValueError(
                'Default configurations are available'
                ' only for the following values of n_blocks: 1, 2, 3, 4, 5, 6.'
                f' However, {n_blocks=}'
            )
        return {
            'n_blocks': n_blocks,
            'd_block': [96, 128, 192, 256, 320, 384][n_blocks - 1],
            'attention_n_heads': 8,
            'attention_dropout': [0.1, 0.15, 0.2, 0.25, 0.3, 0.35][n_blocks - 1],
            'ffn_d_hidden': None,
            # "4 / 3" for ReGLU leads to almost the same number of parameters
            # as "2.0" for ReLU.
            'ffn_d_hidden_multiplier': 4 / 3,
            'ffn_dropout': [0.0, 0.05, 0.1, 0.15, 0.2, 0.25][n_blocks - 1],
            'residual_dropout': 0.0,
            '_is_default': True,
        }

    def make_parameter_groups(self) -> List[Dict[str, Any]]:
        """Make parameter groups for optimizers.

        The difference with calling this method instead of
        `.parameters()` is that this method always sets `weight_decay=0.0`
        for some of the parameters.

        Returns:
            the parameter groups that can be passed to PyTorch optimizers.
        """

        def get_parameters(m: Optional[nn.Module]) -> Iterable[Parameter]:
            return () if m is None else m.parameters()

        zero_wd_group: Dict[str, Any] = {
            'params': set(
                itertools.chain(
                    get_parameters(self.cls_embedding),
                    get_parameters(self.cont_embeddings),
                    get_parameters(self.cat_embeddings),
                    itertools.chain.from_iterable(
                        m.parameters()
                        for block in self.backbone.blocks
                        for name, m in block.named_children()
                        if name.endswith('_normalization')
                    ),
                    (
                        p
                        for name, p in self.named_parameters()
                        if name.endswith('.bias')
                    ),
                )
            ),
            'weight_decay': 0.0,
        }
        main_group: Dict[str, Any] = {
            'params': [p for p in self.parameters() if p not in zero_wd_group['params']]
        }
        zero_wd_group['params'] = list(zero_wd_group['params'])
        return [main_group, zero_wd_group]

    def make_default_optimizer(self) -> torch.optim.AdamW:
        """Create the "default" `torch.nn.AdamW` suitable for the *default* FT-Transformer.

        Returns:
            the optimizer.
        """ # noqa: E501
        if not self._is_default:
            warnings.warn(
                'The default opimizer is supposed to be used in a combination'
                ' with the default FT-Transformer.'
            )
        return torch.optim.AdamW(
            self.make_parameter_groups(), lr=1e-4, weight_decay=1e-5
        )

    _FORWARD_BAD_ARGS_MESSAGE = (
        'Based on the arguments passed to the constructor of FTTransformer, {}'
    )

    def forward(self, x_cont: Optional[Tensor], x_cat: Optional[Tensor]) -> Tensor:
        """Do the forward pass."""
        x_any = x_cat if x_cont is None else x_cont
        if x_any is None:
            raise ValueError('At least one of x_cont and x_cat must be provided.')

        x_embeddings: List[Tensor] = []
        if self.cls_embedding is not None:
            x_embeddings.append(self.cls_embedding(x_any.shape[:-1]))

        for argname, argvalue, module in [
            ('x_cont', x_cont, self.cont_embeddings),
            ('x_cat', x_cat, self.cat_embeddings),
        ]:
            if module is None:
                if argvalue is not None:
                    raise ValueError(
                        FTTransformer._FORWARD_BAD_ARGS_MESSAGE.format(
                            f'{argname} must be None'
                        )
                    )
            else:
                if argvalue is None:
                    raise ValueError(
                        FTTransformer._FORWARD_BAD_ARGS_MESSAGE.format(
                            f'{argname} must not be None'
                        )
                    )
                x_embeddings.append(module(argvalue))
        assert x_embeddings, _INTERNAL_ERROR
        x = torch.cat(x_embeddings, dim=1)
        x = self.backbone(x)
        return x

### 7.2 Train + Evaluate

In [ ]:
trial_results = []
best_auc_global=0.774
global_model_path = str(MODELS_DIR / "ftt_cross_swiglu_hc.pth")
d_out=1
early_stopping = delu.tools.EarlyStopping(patience, mode="max")
d_Block = 64
n_Blocks = 3
Attention_n_heads = 4
FFN_d_hidden = 2
Lr=2e-4
BATCH_SIZE=2048+4096
# Print trial parameters at the start of each trial
print("\n" + "="*70)
print(" Hyperparameters:")
print(f" d_token = {d_Block}")
print(f" n_blocks = {n_Blocks}")
print(f" attention_n_heads = {Attention_n_heads}")
print(f" ffn_d_hidden = {FFN_d_hidden}")
print("="*70 + "\n")
model = FTTransformer(
    n_cont_features=d_numerical,
    cat_cardinalities=cat_cardinalities_final,
    n_blocks=n_Blocks,
    d_block=d_Block,
    attention_n_heads=Attention_n_heads,
    ffn_d_hidden_multiplier=FFN_d_hidden,
    # ffn_activation='SwiGLU',
    attention_dropout=0.05,
    ffn_dropout=0.05,
    residual_dropout=0.05,
    d_out=1 # output a single logit (binary)
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=Lr, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
optimizer,
max_lr=Lr * 5, # Recommended: 3–10× base LR
steps_per_epoch=math.ceil(X_train.shape[0] / BATCH_SIZE), # NUMBER OF BATCHES PER EPOCH
epochs=n_epochs,
pct_start=0.2,
anneal_strategy="cos"
)
batch_size = BATCH_SIZE
epoch_size = math.ceil(X_train.shape[0] / batch_size)
timer = delu.tools.Timer()


n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f" Trainable params: {n_params:,}")

print(f'Test score before training: AUC Val = {evaluate("val"):.4f}')


best = {
    "val": -math.inf,
    "test": -math.inf,
    "epoch": -1,
}

print(f"Device: {device.type.upper()}")
print("-" * 88 + "\n")
timer.run()
for epoch in range(n_epochs):
    pos_weight = torch.tensor(6, device=device)
    neg_weight = torch.tensor(1, device=device)
    for batch in tqdm(
        delu.iter_batches(data["train"], batch_size, shuffle=True),
        desc=f"Epoch {epoch}",
        total=epoch_size,
    ):
        model.train()
        optimizer.zero_grad()

        y_batch = batch["y"].float()
        weights = torch.where(
            y_batch == 1,
            pos_weight,
            neg_weight
        )

        loss = loss_fn(apply_model(batch), batch["y"].float(), weight=weights)
        loss.backward()
        optimizer.step()
        scheduler.step()

    val_auc = evaluate("val")
    test_auc = evaluate("test")
    train_auc = evaluate('train')
    print(f" AUC Train {train_auc:.4f} Val {val_auc:.4f} (test) {test_auc:.4f} [time] {timer}")

    early_stopping.update(val_auc)
    if early_stopping.should_stop():
      print('Early stopping triggered')
      break

    if val_auc > best["val"]:
        print(" New best epoch! ")
        best = {"val": val_auc, "test": test_auc, "epoch": epoch}
        if val_auc>best_auc_global:
            torch.save(model.state_dict(), global_model_path)
            best_auc_global=val_auc
    print()

trial_results.append({
    'type':'SwiGLU in Attention FFN',
    "auc_val": val_auc,
    "auc_test": test_auc,
    "d_token": d_Block,
    "n_blocks": n_Blocks,
    "n_heads": Attention_n_heads,
    "ffn_d_hidden": FFN_d_hidden,
    'best':best,
    'timer':timer.__str__(),
    'train_auc':train_auc
  })

with open(str(RESULTS_DIR / "ftt_cross_swiglu_hc.json"), "w") as f:
    json.dump(trial_results, f, indent=4)

## 8. Variant 4 — Quantile PLE Embedding

Each continuous feature is binned into `n_bins=10` quantile buckets (boundaries computed on train); each bucket has a learned embedding. An interpolation between two adjacent bucket embeddings gives the final token. This is the 'PLE' (Piecewise Linear Encoding) idea from the literature.

### 8.1 Quantile boundary helper

In [ ]:
def compute_quantile_boundaries(X_train, n_bins=10):
    """
    X_train: numpy array or torch tensor of shape [n_samples, n_features]
    Returns: torch tensor of shape [n_features, n_bins + 1]
    """
    if torch.is_tensor(X_train):
        X_train = X_train.detach().cpu().numpy()

    n_features = X_train.shape[1]
    boundaries = []

    # Generate bin edges for each feature independently
    quantiles = np.linspace(0, 1, n_bins + 1)

    for i in range(n_features):
        feature_column = X_train[:, i]
        # Use unique to handle heavily clipped or zero-inflated data
        b = np.unique(np.quantile(feature_column, quantiles))

        # If unique values < n_bins, pad with the last value
        if len(b) < n_bins + 1:
            b = np.concatenate([b, np.full(n_bins + 1 - len(b), b[-1])])

        boundaries.append(b)

    return torch.tensor(np.array(boundaries), dtype=torch.float32)

boundaries_df = compute_quantile_boundaries(data['train']['X_cont'], n_bins=10)

### 8.2 Model

In [ ]:
class QuantilePLEEmbedding(nn.Module):
    def __init__(self, boundaries, d_token):
        """
        boundaries: Tensor of shape [n_features, n_bins + 1]
        """
        super().__init__()
        self.n_features, n_bins_plus_one = boundaries.shape
        self.n_bins = n_bins_plus_one - 1
        self.d_token = d_token

        # Store boundaries so they aren't updated by gradients
        self.register_buffer("boundaries", boundaries)

        # Learnable embeddings for each bin of each feature
        # Shape: [n_features, n_bins, d_token]
        self.embeddings = nn.Parameter(torch.randn(self.n_features, self.n_bins, d_token) * 0.02)

    def forward(self, x):
        # x: [B, n_features]
        B = x.shape[0]

        # 1. Expand x and boundaries for broadcasting
        # x: [B, n_features, 1]
        x_ext = x.unsqueeze(-1)

        # boundaries: [1, n_features, n_bins + 1]
        L = self.boundaries[:, :-1].unsqueeze(0)
        R = self.boundaries[:, 1:].unsqueeze(0)

        # 2. Calculate relative fill of each bin (0.0 to 1.0)
        # Using clamp to ensure values below L are 0 and above R are 1
        # This creates the "Piecewise" effect
        widths = R - L
        # Small epsilon to avoid division by zero in identical quantiles
        widths = torch.clamp(widths, min=1e-6)

        weights = (x_ext - L) / widths
        weights = torch.clamp(weights, 0.0, 1.0) # [B, n_features, n_bins]

        # 3. Multiply weights by bin-specific embeddings
        # [B, n_features, n_bins, 1] * [1, n_features, n_bins, d_token]
        out = weights.unsqueeze(-1) * self.embeddings.unsqueeze(0)

        # 4. Sum across bins to get the final token for each feature
        return out.sum(dim=2) # Result: [B, n_features, d_token]


class FTTransformer(nn.Module):
    """The FT-Transformer model from Section 3.3 in the paper."""

    def __init__(
        self,
        *,
        n_cont_features: int,
        cat_cardinalities: List[int],
        boundaries: Optional[Tensor] = None,
        _is_default: bool = False,
        **backbone_kwargs,
    ) -> None:
        if n_cont_features < 0:
            raise ValueError(
                f'n_cont_features must be non-negative, however: {n_cont_features=}'
            )
        if n_cont_features == 0 and not cat_cardinalities:
            raise ValueError(
                'At least one type of features must be presented, however:'
                f' {n_cont_features=}, {cat_cardinalities=}'
            )
        if 'n_tokens' in backbone_kwargs:
            raise ValueError(
                'backbone_kwargs must not contain key "n_tokens"'
                ' (the number of tokens will be inferred automatically)'
            )

        super().__init__()
        d_block: int = backbone_kwargs['d_block']
        self.cls_embedding = _CLSEmbedding(d_block)

        # >>> Feature embeddings (Figure 2a in the paper).
        # self.cont_embeddings = (
        # # QuantilePLEEmbedding(boundaries, d_block) if n_cont_features > 0 else None
        # # LinearEmbeddings(n_cont_features, d_block) if n_cont_features > 0 else None
        # )

        if n_cont_features > 0:
            self.linear_embeddings = LinearEmbeddings(n_cont_features, d_block)
            # self.periodic_embeddings = PeriodicEmbedding(n_cont_features, d_block)
            self.ple_embeddings = QuantilePLEEmbedding(boundaries, d_block)
            self.cont_norm = nn.LayerNorm(d_block)
            # self.cont_embeddings = self.linear_embeddings+self.ple_embeddings
        else: None

        self.cat_embeddings = (
            CategoricalEmbeddings(cat_cardinalities, d_block, True)
            if cat_cardinalities
            else None
        )
        # <<<

        self.backbone = FTTransformerBackbone(
            **backbone_kwargs,
            n_tokens=(
                None
                if backbone_kwargs.get('linformer_kv_compression_ratio') is None
                else 1 + n_cont_features + len(cat_cardinalities)
            ),
        )
        self._is_default = _is_default

    @classmethod
    def get_default_kwargs(cls, n_blocks: int = 3) -> Dict[str, Any]:
        """Get the default hyperparameters.

        Args:
            n_blocks: the number of blocks. The supported values are: 1, 2, 3, 4, 5, 6.
        Returns:
            the default keyword arguments for the constructor.
        """
        if n_blocks < 0 or n_blocks > 6:
            raise ValueError(
                'Default configurations are available'
                ' only for the following values of n_blocks: 1, 2, 3, 4, 5, 6.'
                f' However, {n_blocks=}'
            )
        return {
            'n_blocks': n_blocks,
            'd_block': [96, 128, 192, 256, 320, 384][n_blocks - 1],
            'attention_n_heads': 8,
            'attention_dropout': [0.1, 0.15, 0.2, 0.25, 0.3, 0.35][n_blocks - 1],
            'ffn_d_hidden': None,
            # "4 / 3" for ReGLU leads to almost the same number of parameters
            # as "2.0" for ReLU.
            'ffn_d_hidden_multiplier': 4 / 3,
            'ffn_dropout': [0.0, 0.05, 0.1, 0.15, 0.2, 0.25][n_blocks - 1],
            'residual_dropout': 0.0,
            '_is_default': True,
        }

    def make_parameter_groups(self) -> List[Dict[str, Any]]:
        def get_parameters(m: Optional[nn.Module]) -> Iterable[Parameter]:
            return () if m is None else m.parameters()

        zero_wd_group: Dict[str, Any] = {
            'params': set(
                itertools.chain(
                    get_parameters(self.cls_embedding),
                    get_parameters(self.cont_embeddings),
                    get_parameters(self.cat_embeddings),
                    itertools.chain.from_iterable(
                        m.parameters()
                        for block in self.backbone.blocks
                        for name, m in block.named_children()
                        if name.endswith('_normalization')
                    ),
                    (
                        p
                        for name, p in self.named_parameters()
                        if name.endswith('.bias')
                    ),
                )
            ),
            'weight_decay': 0.0,
        }
        main_group: Dict[str, Any] = {
            'params': [p for p in self.parameters() if p not in zero_wd_group['params']]
        }
        zero_wd_group['params'] = list(zero_wd_group['params'])
        return [main_group, zero_wd_group]

    def make_default_optimizer(self) -> torch.optim.AdamW:
        """Create the "default" `torch.nn.AdamW` suitable for the *default* FT-Transformer.

        Returns:
            the optimizer.
        """ # noqa: E501
        if not self._is_default:
            warnings.warn(
                'The default opimizer is supposed to be used in a combination'
                ' with the default FT-Transformer.'
            )
        return torch.optim.AdamW(
            self.make_parameter_groups(), lr=1e-4, weight_decay=1e-5
        )

    _FORWARD_BAD_ARGS_MESSAGE = (
        'Based on the arguments passed to the constructor of FTTransformer, {}'
    )

    def forward(self, x_cont: Optional[Tensor], x_cat: Optional[Tensor]) -> Tensor:
        """Do the forward pass."""
        x_any = x_cat if x_cont is None else x_cont
        if x_any is None:
            raise ValueError('At least one of x_cont and x_cat must be provided.')

        x_embeddings: List[Tensor] = []
        if self.cls_embedding is not None:
            x_embeddings.append(self.cls_embedding(x_any.shape[:-1]))

        if x_cont is not None and self.ple_embeddings is not None:
        # if x_cont is not None:
            # 1. Get Global Linear Trend
            # lin_tokens = self.linear_embeddings(x_cont)
            # per_tokens = self.periodic_embeddings(x_cont)
            ple_tokens = self.ple_embeddings(x_cont)
            x_embeddings.append(ple_tokens)
            # x_embeddings.append(self.cont_norm(lin_tokens + ple_tokens))

        # Handle Categorical Features
        if x_cat is not None and self.cat_embeddings is not None:
            x_embeddings.append(self.cat_embeddings(x_cat))

        assert x_embeddings, _INTERNAL_ERROR
        x = torch.cat(x_embeddings, dim=1)
        x = self.backbone(x)
        return x

### 8.3 Train + Evaluate

In [ ]:
trial_results = []
best_auc_global=0.774
global_model_path = str(MODELS_DIR / "ftt_qple_hc.pth")
d_out=1
early_stopping = delu.tools.EarlyStopping(patience, mode="max")
d_Block = 64
n_Blocks = 3
Attention_n_heads = 4
FFN_d_hidden = 2
Lr=2e-4
BATCH_SIZE=1024+4096
# Print trial parameters at the start of each trial
print("\n" + "="*70)
print(" Hyperparameters:")
print(f" d_token = {d_Block}")
print(f" n_blocks = {n_Blocks}")
print(f" attention_n_heads = {Attention_n_heads}")
print(f" ffn_d_hidden = {FFN_d_hidden}")
print("="*70 + "\n")
model = FTTransformer(
    n_cont_features=d_numerical,
    cat_cardinalities=cat_cardinalities_final,
    n_blocks=n_Blocks,
    d_block=d_Block,
    boundaries = boundaries_df,
    attention_n_heads=Attention_n_heads,
    ffn_d_hidden_multiplier=FFN_d_hidden,
    # ffn_activation='SwiGLU',
    attention_dropout=0.05,
    ffn_dropout=0.05,
    residual_dropout=0.05,
    d_out=1 # output a single logit (binary)
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=Lr, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
optimizer,
max_lr=Lr * 5, # Recommended: 3–10× base LR
steps_per_epoch=math.ceil(X_train.shape[0] / BATCH_SIZE), # NUMBER OF BATCHES PER EPOCH
epochs=n_epochs,
pct_start=0.2,
anneal_strategy="cos"
)
batch_size = BATCH_SIZE
epoch_size = math.ceil(X_train.shape[0] / batch_size)
timer = delu.tools.Timer()


n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f" Trainable params: {n_params:,}")

print(f'Test score before training: AUC Val = {evaluate("val"):.4f}')


best = {
    "val": -math.inf,
    "test": -math.inf,
    "epoch": -1,
}

print(f"Device: {device.type.upper()}")
print("-" * 88 + "\n")
timer.run()
for epoch in range(n_epochs):
    pos_weight = torch.tensor(6, device=device)
    neg_weight = torch.tensor(1, device=device)
    for batch in tqdm(
        delu.iter_batches(data["train"], batch_size, shuffle=True),
        desc=f"Epoch {epoch}",
        total=epoch_size,
    ):
        model.train()
        optimizer.zero_grad()

        y_batch = batch["y"].float()
        weights = torch.where(
            y_batch == 1,
            pos_weight,
            neg_weight
        )

        loss = loss_fn(apply_model(batch), batch["y"].float(), weight=weights)
        loss.backward()
        optimizer.step()
        scheduler.step()

    val_auc = evaluate("val")
    test_auc = evaluate("test")
    train_auc = evaluate('train')
    print(f" AUC Train {train_auc:.4f} Val {val_auc:.4f} (test) {test_auc:.4f} [time] {timer}")

    early_stopping.update(val_auc)
    if early_stopping.should_stop():
      print('Early stopping triggered')
      break

    if val_auc > best["val"]:
        print(" New best epoch! ")
        best = {"val": val_auc, "test": test_auc, "epoch": epoch}
        if val_auc>best_auc_global:
            torch.save(model.state_dict(), global_model_path)
            best_auc_global=val_auc
    print()

trial_results.append({
    'type':'SwiGLU in Attention FFN',
    "auc_val": val_auc,
    "auc_test": test_auc,
    "d_token": d_Block,
    "n_blocks": n_Blocks,
    "n_heads": Attention_n_heads,
    "ffn_d_hidden": FFN_d_hidden,
    'best':best,
    'timer':timer.__str__(),
    'train_auc':train_auc
  })

with open(str(RESULTS_DIR / "ftt_qple_hc.json"), "w") as f:
    json.dump(trial_results, f, indent=4)

## 9. Variant 5 — QPLE + Linear

Same as Variant 4 but concatenates the QPLE embedding with a standard linear embedding before passing into the transformer backbone. The linear branch acts as a residual that prevents the quantile binning from throwing away within-bucket information.

### 9.1 Model

In [ ]:
class QuantilePLEEmbedding(nn.Module):
    def __init__(self, boundaries, d_token):
        """
        boundaries: Tensor of shape [n_features, n_bins + 1]
        """
        super().__init__()
        self.n_features, n_bins_plus_one = boundaries.shape
        self.n_bins = n_bins_plus_one - 1
        self.d_token = d_token

        # Store boundaries so they aren't updated by gradients
        self.register_buffer("boundaries", boundaries)

        # Learnable embeddings for each bin of each feature
        # Shape: [n_features, n_bins, d_token]
        self.embeddings = nn.Parameter(torch.randn(self.n_features, self.n_bins, d_token) * 0.02)

    def forward(self, x):
        # x: [B, n_features]
        B = x.shape[0]

        # 1. Expand x and boundaries for broadcasting
        # x: [B, n_features, 1]
        x_ext = x.unsqueeze(-1)

        # boundaries: [1, n_features, n_bins + 1]
        L = self.boundaries[:, :-1].unsqueeze(0)
        R = self.boundaries[:, 1:].unsqueeze(0)

        # 2. Calculate relative fill of each bin (0.0 to 1.0)
        # Using clamp to ensure values below L are 0 and above R are 1
        # This creates the "Piecewise" effect
        widths = R - L
        # Small epsilon to avoid division by zero in identical quantiles
        widths = torch.clamp(widths, min=1e-6)

        weights = (x_ext - L) / widths
        weights = torch.clamp(weights, 0.0, 1.0) # [B, n_features, n_bins]

        # 3. Multiply weights by bin-specific embeddings
        # [B, n_features, n_bins, 1] * [1, n_features, n_bins, d_token]
        out = weights.unsqueeze(-1) * self.embeddings.unsqueeze(0)

        # 4. Sum across bins to get the final token for each feature
        return out.sum(dim=2) # Result: [B, n_features, d_token]


class FTTransformer(nn.Module):
    """The FT-Transformer model from Section 3.3 in the paper."""

    def __init__(
        self,
        *,
        n_cont_features: int,
        cat_cardinalities: List[int],
        boundaries: Optional[Tensor] = None,
        _is_default: bool = False,
        **backbone_kwargs,
    ) -> None:
        if n_cont_features < 0:
            raise ValueError(
                f'n_cont_features must be non-negative, however: {n_cont_features=}'
            )
        if n_cont_features == 0 and not cat_cardinalities:
            raise ValueError(
                'At least one type of features must be presented, however:'
                f' {n_cont_features=}, {cat_cardinalities=}'
            )
        if 'n_tokens' in backbone_kwargs:
            raise ValueError(
                'backbone_kwargs must not contain key "n_tokens"'
                ' (the number of tokens will be inferred automatically)'
            )

        super().__init__()
        d_block: int = backbone_kwargs['d_block']
        self.cls_embedding = _CLSEmbedding(d_block)

        # >>> Feature embeddings (Figure 2a in the paper).
        # self.cont_embeddings = (
        # # QuantilePLEEmbedding(boundaries, d_block) if n_cont_features > 0 else None
        # # LinearEmbeddings(n_cont_features, d_block) if n_cont_features > 0 else None
        # )

        if n_cont_features > 0:
            self.linear_embeddings = LinearEmbeddings(n_cont_features, d_block)
            # self.periodic_embeddings = PeriodicEmbedding(n_cont_features, d_block)
            self.ple_embeddings = QuantilePLEEmbedding(boundaries, d_block)
            self.cont_norm = nn.LayerNorm(d_block)
            # self.cont_embeddings = self.linear_embeddings+self.ple_embeddings
        else: None

        self.cat_embeddings = (
            CategoricalEmbeddings(cat_cardinalities, d_block, True)
            if cat_cardinalities
            else None
        )
        # <<<

        self.backbone = FTTransformerBackbone(
            **backbone_kwargs,
            n_tokens=(
                None
                if backbone_kwargs.get('linformer_kv_compression_ratio') is None
                else 1 + n_cont_features + len(cat_cardinalities)
            ),
        )
        self._is_default = _is_default

    @classmethod
    def get_default_kwargs(cls, n_blocks: int = 3) -> Dict[str, Any]:
        """Get the default hyperparameters.

        Args:
            n_blocks: the number of blocks. The supported values are: 1, 2, 3, 4, 5, 6.
        Returns:
            the default keyword arguments for the constructor.
        """
        if n_blocks < 0 or n_blocks > 6:
            raise ValueError(
                'Default configurations are available'
                ' only for the following values of n_blocks: 1, 2, 3, 4, 5, 6.'
                f' However, {n_blocks=}'
            )
        return {
            'n_blocks': n_blocks,
            'd_block': [96, 128, 192, 256, 320, 384][n_blocks - 1],
            'attention_n_heads': 8,
            'attention_dropout': [0.1, 0.15, 0.2, 0.25, 0.3, 0.35][n_blocks - 1],
            'ffn_d_hidden': None,
            # "4 / 3" for ReGLU leads to almost the same number of parameters
            # as "2.0" for ReLU.
            'ffn_d_hidden_multiplier': 4 / 3,
            'ffn_dropout': [0.0, 0.05, 0.1, 0.15, 0.2, 0.25][n_blocks - 1],
            'residual_dropout': 0.0,
            '_is_default': True,
        }

    def make_parameter_groups(self) -> List[Dict[str, Any]]:
        def get_parameters(m: Optional[nn.Module]) -> Iterable[Parameter]:
            return () if m is None else m.parameters()

        zero_wd_group: Dict[str, Any] = {
            'params': set(
                itertools.chain(
                    get_parameters(self.cls_embedding),
                    get_parameters(self.cont_embeddings),
                    get_parameters(self.cat_embeddings),
                    itertools.chain.from_iterable(
                        m.parameters()
                        for block in self.backbone.blocks
                        for name, m in block.named_children()
                        if name.endswith('_normalization')
                    ),
                    (
                        p
                        for name, p in self.named_parameters()
                        if name.endswith('.bias')
                    ),
                )
            ),
            'weight_decay': 0.0,
        }
        main_group: Dict[str, Any] = {
            'params': [p for p in self.parameters() if p not in zero_wd_group['params']]
        }
        zero_wd_group['params'] = list(zero_wd_group['params'])
        return [main_group, zero_wd_group]

    def make_default_optimizer(self) -> torch.optim.AdamW:
        """Create the "default" `torch.nn.AdamW` suitable for the *default* FT-Transformer.

        Returns:
            the optimizer.
        """ # noqa: E501
        if not self._is_default:
            warnings.warn(
                'The default opimizer is supposed to be used in a combination'
                ' with the default FT-Transformer.'
            )
        return torch.optim.AdamW(
            self.make_parameter_groups(), lr=1e-4, weight_decay=1e-5
        )

    _FORWARD_BAD_ARGS_MESSAGE = (
        'Based on the arguments passed to the constructor of FTTransformer, {}'
    )

    def forward(self, x_cont: Optional[Tensor], x_cat: Optional[Tensor]) -> Tensor:
        """Do the forward pass."""
        x_any = x_cat if x_cont is None else x_cont
        if x_any is None:
            raise ValueError('At least one of x_cont and x_cat must be provided.')

        x_embeddings: List[Tensor] = []
        if self.cls_embedding is not None:
            x_embeddings.append(self.cls_embedding(x_any.shape[:-1]))

        if x_cont is not None and self.ple_embeddings is not None:
        # if x_cont is not None:
            # 1. Get Global Linear Trend
            lin_tokens = self.linear_embeddings(x_cont)
            # per_tokens = self.periodic_embeddings(x_cont)
            ple_tokens = self.ple_embeddings(x_cont)
            # x_embeddings.append(ple_tokens)
            x_embeddings.append(self.cont_norm(lin_tokens + ple_tokens))

        # Handle Categorical Features
        if x_cat is not None and self.cat_embeddings is not None:
            x_embeddings.append(self.cat_embeddings(x_cat))

        assert x_embeddings, _INTERNAL_ERROR
        x = torch.cat(x_embeddings, dim=1)
        x = self.backbone(x)
        return x

### 9.2 Train + Evaluate

In [ ]:
trial_results = []
best_auc_global=0.774
global_model_path = str(MODELS_DIR / "ftt_qple_linear_hc.pth")
d_out=1
early_stopping = delu.tools.EarlyStopping(patience, mode="max")
d_Block = 64
n_Blocks = 3
Attention_n_heads = 4
FFN_d_hidden = 2
Lr=2e-4
BATCH_SIZE=1024+4096
# Print trial parameters at the start of each trial
print("\n" + "="*70)
print(" Hyperparameters:")
print(f" d_token = {d_Block}")
print(f" n_blocks = {n_Blocks}")
print(f" attention_n_heads = {Attention_n_heads}")
print(f" ffn_d_hidden = {FFN_d_hidden}")
print("="*70 + "\n")
model = FTTransformer(
    n_cont_features=d_numerical,
    cat_cardinalities=cat_cardinalities_final,
    n_blocks=n_Blocks,
    d_block=d_Block,
    boundaries = boundaries_df,
    attention_n_heads=Attention_n_heads,
    ffn_d_hidden_multiplier=FFN_d_hidden,
    # ffn_activation='SwiGLU',
    attention_dropout=0.05,
    ffn_dropout=0.05,
    residual_dropout=0.05,
    d_out=1 # output a single logit (binary)
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=Lr, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
optimizer,
max_lr=Lr * 5, # Recommended: 3–10× base LR
steps_per_epoch=math.ceil(X_train.shape[0] / BATCH_SIZE), # NUMBER OF BATCHES PER EPOCH
epochs=n_epochs,
pct_start=0.2,
anneal_strategy="cos"
)
batch_size = BATCH_SIZE
epoch_size = math.ceil(X_train.shape[0] / batch_size)
timer = delu.tools.Timer()


n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f" Trainable params: {n_params:,}")

print(f'Test score before training: AUC Val = {evaluate("val"):.4f}')


best = {
    "val": -math.inf,
    "test": -math.inf,
    "epoch": -1,
}

print(f"Device: {device.type.upper()}")
print("-" * 88 + "\n")
timer.run()
for epoch in range(n_epochs):
    pos_weight = torch.tensor(6, device=device)
    neg_weight = torch.tensor(1, device=device)
    for batch in tqdm(
        delu.iter_batches(data["train"], batch_size, shuffle=True),
        desc=f"Epoch {epoch}",
        total=epoch_size,
    ):
        model.train()
        optimizer.zero_grad()

        y_batch = batch["y"].float()
        weights = torch.where(
            y_batch == 1,
            pos_weight,
            neg_weight
        )

        loss = loss_fn(apply_model(batch), batch["y"].float(), weight=weights)
        loss.backward()
        optimizer.step()
        scheduler.step()

    val_auc = evaluate("val")
    test_auc = evaluate("test")
    train_auc = evaluate('train')
    print(f" AUC Train {train_auc:.4f} Val {val_auc:.4f} (test) {test_auc:.4f} [time] {timer}")

    early_stopping.update(val_auc)
    if early_stopping.should_stop():
      print('Early stopping triggered')
      break

    if val_auc > best["val"]:
        print(" New best epoch! ")
        best = {"val": val_auc, "test": test_auc, "epoch": epoch}
        if val_auc>best_auc_global:
            torch.save(model.state_dict(), global_model_path)
            best_auc_global=val_auc
    print()

trial_results.append({
    'type':'SwiGLU in Attention FFN',
    "auc_val": val_auc,
    "auc_test": test_auc,
    "d_token": d_Block,
    "n_blocks": n_Blocks,
    "n_heads": Attention_n_heads,
    "ffn_d_hidden": FFN_d_hidden,
    'best':best,
    'timer':timer.__str__(),
    'train_auc':train_auc
  })

with open(str(RESULTS_DIR / "ftt_qple_linear_hc.json"), "w") as f:
    json.dump(trial_results, f, indent=4)

## 10. Variant 6 — QPLE + Linear + Periodic

Stacks a periodic (sin/cos) embedding on top of Variant 5. The periodic features give the model an explicit way to represent fine-grained continuous patterns that may be lost in the quantile bins.

### 10.1 Model

In [ ]:
from typing import Union
class _Periodic(nn.Module):
    """
    NOTE: THIS MODULE SHOULD NOT BE USED DIRECTLY.

    Technically, this is a linear embedding without bias followed by
    the periodic activations. The scale of the initialization
    (defined by the `sigma` argument) plays an important role.
    """

    def __init__(self, n_features: int, k: int, sigma: float) -> None:
        if sigma <= 0.0:
            raise ValueError(f'sigma must be positive, however: {sigma=}')

        super().__init__()
        self._sigma = sigma
        self.weight = Parameter(torch.empty(n_features, k))
        self.reset_parameters()

    def reset_parameters(self):
        """Reset the parameters."""
        # NOTE[DIFF]
        # Here, extreme values (~0.3% probability) are explicitly avoided just in case.
        # In the paper, there was no protection from extreme values.
        bound = self._sigma * 3
        nn.init.trunc_normal_(self.weight, 0.0, self._sigma, a=-bound, b=bound)

    def forward(self, x: Tensor) -> Tensor:
        """Do the forward pass."""
        _check_input_shape(x, self.weight.shape[0])
        x = 2 * math.pi * self.weight * x[..., None]
        x = torch.cat([torch.cos(x), torch.sin(x)], -1)
        return x

class _NLinear(nn.Module):
    """N *separate* linear layers for N feature embeddings.

    In other words,
    each feature embedding is transformed by its own dedicated linear layer.
    """

    def __init__(
        self, n: int, in_features: int, out_features: int, bias: bool = True
    ) -> None:
        super().__init__()
        self.weight = Parameter(torch.empty(n, in_features, out_features))
        self.bias = Parameter(torch.empty(n, out_features)) if bias else None
        self.reset_parameters()

    def reset_parameters(self):
        """Reset the parameters."""
        d_in_rsqrt = self.weight.shape[-2] ** -0.5
        nn.init.uniform_(self.weight, -d_in_rsqrt, d_in_rsqrt)
        if self.bias is not None:
            nn.init.uniform_(self.bias, -d_in_rsqrt, d_in_rsqrt)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Do the forward pass."""
        if x.ndim != 3:
            raise ValueError(
                '_NLinear supports only inputs with exactly one batch dimension,'
                ' so `x` must have a shape like (BATCH_SIZE, N_FEATURES, D_EMBEDDING).'
            )
        assert x.shape[-(self.weight.ndim - 1) :] == self.weight.shape[:-1]

        x = x.transpose(0, 1)
        x = x @ self.weight
        x = x.transpose(0, 1)
        if self.bias is not None:
            x = x + self.bias
        return x

class PeriodicEmbeddings(nn.Module):
      def __init__(
        self,
        n_features: int,
        d_embedding: int = 24,
        *,
        n_frequencies: int = 48,
        frequency_init_scale: float = 0.01,
        activation: bool = True,
        lite: bool=False,
    ) -> None:
        """
        Args:
            n_features: the number of features.
            d_embedding: the embedding size.
            n_frequencies: the number of frequencies for each feature.
                (denoted as "k" in Section 3.3 in the paper).
            frequency_init_scale: the initialization scale for the first linear layer
                (denoted as "sigma" in Section 3.3 in the paper).
                **This is an important hyperparameter**, see README for details.
            activation: if `False`, the ReLU activation is not applied.
                Must be `True` if ``lite=True``.
            lite: if True, the outer linear layer is shared between all features.
                See README for details.
        """
        super().__init__()
        self.periodic = _Periodic(n_features, n_frequencies, frequency_init_scale)
        self.linear: Union[nn.Linear, _NLinear]
        if lite:
            # NOTE[DIFF]
            # The lite variation was introduced in a different paper
            # (about the TabR model).
            if not activation:
                raise ValueError('lite=True is allowed only when activation=True')
            self.linear = nn.Linear(2 * n_frequencies, d_embedding)
        else:
            self.linear = _NLinear(n_features, 2 * n_frequencies, d_embedding)
        self.activation = nn.ReLU() if activation else None

      def get_output_shape(self) -> torch.Size:
          """Get the output shape without the batch dimensions."""
          n_features = self.periodic.weight.shape[0]
          d_embedding = (
              self.linear.weight.shape[0]
              if isinstance(self.linear, nn.Linear)
              else self.linear.weight.shape[-1]
          )
          return torch.Size((n_features, d_embedding))

      def forward(self, x: Tensor) -> Tensor:
          """Do the forward pass."""
          x = self.periodic(x)
          x = self.linear(x)
          if self.activation is not None:
              x = self.activation(x)
          return x


class QuantilePLEEmbedding(nn.Module):
    def __init__(self, boundaries, d_token):
        """
        boundaries: Tensor of shape [n_features, n_bins + 1]
        """
        super().__init__()
        self.n_features, n_bins_plus_one = boundaries.shape
        self.n_bins = n_bins_plus_one - 1
        self.d_token = d_token

        # Store boundaries so they aren't updated by gradients
        self.register_buffer("boundaries", boundaries)

        # Learnable embeddings for each bin of each feature
        # Shape: [n_features, n_bins, d_token]
        self.embeddings = nn.Parameter(torch.randn(self.n_features, self.n_bins, d_token) * 0.02)

    def forward(self, x):
        # x: [B, n_features]
        B = x.shape[0]

        # 1. Expand x and boundaries for broadcasting
        # x: [B, n_features, 1]
        x_ext = x.unsqueeze(-1)

        # boundaries: [1, n_features, n_bins + 1]
        L = self.boundaries[:, :-1].unsqueeze(0)
        R = self.boundaries[:, 1:].unsqueeze(0)

        # 2. Calculate relative fill of each bin (0.0 to 1.0)
        # Using clamp to ensure values below L are 0 and above R are 1
        # This creates the "Piecewise" effect
        widths = R - L
        # Small epsilon to avoid division by zero in identical quantiles
        widths = torch.clamp(widths, min=1e-6)

        weights = (x_ext - L) / widths
        weights = torch.clamp(weights, 0.0, 1.0) # [B, n_features, n_bins]

        # 3. Multiply weights by bin-specific embeddings
        # [B, n_features, n_bins, 1] * [1, n_features, n_bins, d_token]
        out = weights.unsqueeze(-1) * self.embeddings.unsqueeze(0)

        # 4. Sum across bins to get the final token for each feature
        return out.sum(dim=2) # Result: [B, n_features, d_token]


class FTTransformer(nn.Module):
    """The FT-Transformer model from Section 3.3 in the paper."""

    def __init__(
        self,
        *,
        n_cont_features: int,
        cat_cardinalities: List[int],
        boundaries: Optional[Tensor] = None,
        _is_default: bool = False,
        **backbone_kwargs,
    ) -> None:
        if n_cont_features < 0:
            raise ValueError(
                f'n_cont_features must be non-negative, however: {n_cont_features=}'
            )
        if n_cont_features == 0 and not cat_cardinalities:
            raise ValueError(
                'At least one type of features must be presented, however:'
                f' {n_cont_features=}, {cat_cardinalities=}'
            )
        if 'n_tokens' in backbone_kwargs:
            raise ValueError(
                'backbone_kwargs must not contain key "n_tokens"'
                ' (the number of tokens will be inferred automatically)'
            )

        super().__init__()
        d_block: int = backbone_kwargs['d_block']
        self.cls_embedding = _CLSEmbedding(d_block)

        # >>> Feature embeddings (Figure 2a in the paper).
        # self.cont_embeddings = (
        # # QuantilePLEEmbedding(boundaries, d_block) if n_cont_features > 0 else None
        # # LinearEmbeddings(n_cont_features, d_block) if n_cont_features > 0 else None
        # )

        if n_cont_features > 0:
            self.linear_embeddings = LinearEmbeddings(n_cont_features, d_block)
            self.periodic_embeddings = PeriodicEmbeddings(n_cont_features, d_block)
            self.ple_embeddings = QuantilePLEEmbedding(boundaries, d_block)
            self.cont_norm = nn.LayerNorm(d_block)
            # self.cont_embeddings = self.linear_embeddings+self.ple_embeddings
        else: None

        self.cat_embeddings = (
            CategoricalEmbeddings(cat_cardinalities, d_block, True)
            if cat_cardinalities
            else None
        )
        # <<<

        self.backbone = FTTransformerBackbone(
            **backbone_kwargs,
            n_tokens=(
                None
                if backbone_kwargs.get('linformer_kv_compression_ratio') is None
                else 1 + n_cont_features + len(cat_cardinalities)
            ),
        )
        self._is_default = _is_default

    @classmethod
    def get_default_kwargs(cls, n_blocks: int = 3) -> Dict[str, Any]:
        """Get the default hyperparameters.

        Args:
            n_blocks: the number of blocks. The supported values are: 1, 2, 3, 4, 5, 6.
        Returns:
            the default keyword arguments for the constructor.
        """
        if n_blocks < 0 or n_blocks > 6:
            raise ValueError(
                'Default configurations are available'
                ' only for the following values of n_blocks: 1, 2, 3, 4, 5, 6.'
                f' However, {n_blocks=}'
            )
        return {
            'n_blocks': n_blocks,
            'd_block': [96, 128, 192, 256, 320, 384][n_blocks - 1],
            'attention_n_heads': 8,
            'attention_dropout': [0.1, 0.15, 0.2, 0.25, 0.3, 0.35][n_blocks - 1],
            'ffn_d_hidden': None,
            # "4 / 3" for ReGLU leads to almost the same number of parameters
            # as "2.0" for ReLU.
            'ffn_d_hidden_multiplier': 4 / 3,
            'ffn_dropout': [0.0, 0.05, 0.1, 0.15, 0.2, 0.25][n_blocks - 1],
            'residual_dropout': 0.0,
            '_is_default': True,
        }

    def make_parameter_groups(self) -> List[Dict[str, Any]]:
        def get_parameters(m: Optional[nn.Module]) -> Iterable[Parameter]:
            return () if m is None else m.parameters()

        zero_wd_group: Dict[str, Any] = {
            'params': set(
                itertools.chain(
                    get_parameters(self.cls_embedding),
                    get_parameters(self.cont_embeddings),
                    get_parameters(self.cat_embeddings),
                    itertools.chain.from_iterable(
                        m.parameters()
                        for block in self.backbone.blocks
                        for name, m in block.named_children()
                        if name.endswith('_normalization')
                    ),
                    (
                        p
                        for name, p in self.named_parameters()
                        if name.endswith('.bias')
                    ),
                )
            ),
            'weight_decay': 0.0,
        }
        main_group: Dict[str, Any] = {
            'params': [p for p in self.parameters() if p not in zero_wd_group['params']]
        }
        zero_wd_group['params'] = list(zero_wd_group['params'])
        return [main_group, zero_wd_group]

    def make_default_optimizer(self) -> torch.optim.AdamW:
        """Create the "default" `torch.nn.AdamW` suitable for the *default* FT-Transformer.

        Returns:
            the optimizer.
        """ # noqa: E501
        if not self._is_default:
            warnings.warn(
                'The default opimizer is supposed to be used in a combination'
                ' with the default FT-Transformer.'
            )
        return torch.optim.AdamW(
            self.make_parameter_groups(), lr=1e-4, weight_decay=1e-5
        )

    _FORWARD_BAD_ARGS_MESSAGE = (
        'Based on the arguments passed to the constructor of FTTransformer, {}'
    )

    def forward(self, x_cont: Optional[Tensor], x_cat: Optional[Tensor]) -> Tensor:
        """Do the forward pass."""
        x_any = x_cat if x_cont is None else x_cont
        if x_any is None:
            raise ValueError('At least one of x_cont and x_cat must be provided.')

        x_embeddings: List[Tensor] = []
        if self.cls_embedding is not None:
            x_embeddings.append(self.cls_embedding(x_any.shape[:-1]))

        if x_cont is not None and self.ple_embeddings is not None:
        # if x_cont is not None:
            # 1. Get Global Linear Trend
            lin_tokens = self.linear_embeddings(x_cont)
            per_tokens = self.periodic_embeddings(x_cont)
            ple_tokens = self.ple_embeddings(x_cont)
            # x_embeddings.append(ple_tokens)
            x_embeddings.append(self.cont_norm(lin_tokens + ple_tokens+per_tokens))

        # Handle Categorical Features
        if x_cat is not None and self.cat_embeddings is not None:
            x_embeddings.append(self.cat_embeddings(x_cat))

        assert x_embeddings, _INTERNAL_ERROR
        x = torch.cat(x_embeddings, dim=1)
        x = self.backbone(x)
        return x

### 10.2 Train + Evaluate

In [ ]:
trial_results = []
best_auc_global=0.774
global_model_path = str(MODELS_DIR / "ftt_qple_linear_periodic_hc.pth")
d_out=1
early_stopping = delu.tools.EarlyStopping(patience, mode="max")
d_Block = 64
n_Blocks = 3
Attention_n_heads = 4
FFN_d_hidden = 2
Lr=2e-4
BATCH_SIZE=1024+4096
# Print trial parameters at the start of each trial
print("\n" + "="*70)
print(" Hyperparameters:")
print(f" d_token = {d_Block}")
print(f" n_blocks = {n_Blocks}")
print(f" attention_n_heads = {Attention_n_heads}")
print(f" ffn_d_hidden = {FFN_d_hidden}")
print("="*70 + "\n")
model = FTTransformer(
    n_cont_features=d_numerical,
    cat_cardinalities=cat_cardinalities_final,
    n_blocks=n_Blocks,
    d_block=d_Block,
    boundaries = boundaries_df,
    attention_n_heads=Attention_n_heads,
    ffn_d_hidden_multiplier=FFN_d_hidden,
    # ffn_activation='SwiGLU',
    attention_dropout=0.05,
    ffn_dropout=0.05,
    residual_dropout=0.05,
    d_out=1 # output a single logit (binary)
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=Lr, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
optimizer,
max_lr=Lr * 5, # Recommended: 3–10× base LR
steps_per_epoch=math.ceil(X_train.shape[0] / BATCH_SIZE), # NUMBER OF BATCHES PER EPOCH
epochs=n_epochs,
pct_start=0.2,
anneal_strategy="cos"
)
batch_size = BATCH_SIZE
epoch_size = math.ceil(X_train.shape[0] / batch_size)
timer = delu.tools.Timer()


n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f" Trainable params: {n_params:,}")

print(f'Test score before training: AUC Val = {evaluate("val"):.4f}')


best = {
    "val": -math.inf,
    "test": -math.inf,
    "epoch": -1,
}

print(f"Device: {device.type.upper()}")
print("-" * 88 + "\n")
timer.run()
for epoch in range(n_epochs):
    pos_weight = torch.tensor(6, device=device)
    neg_weight = torch.tensor(1, device=device)
    for batch in tqdm(
        delu.iter_batches(data["train"], batch_size, shuffle=True),
        desc=f"Epoch {epoch}",
        total=epoch_size,
    ):
        model.train()
        optimizer.zero_grad()

        y_batch = batch["y"].float()
        weights = torch.where(
            y_batch == 1,
            pos_weight,
            neg_weight
        )

        loss = loss_fn(apply_model(batch), batch["y"].float(), weight=weights)
        loss.backward()
        optimizer.step()
        scheduler.step()

    val_auc = evaluate("val")
    test_auc = evaluate("test")
    train_auc = evaluate('train')
    print(f" AUC Train {train_auc:.4f} Val {val_auc:.4f} (test) {test_auc:.4f} [time] {timer}")

    early_stopping.update(val_auc)
    if early_stopping.should_stop():
      print('Early stopping triggered')
      break

    if val_auc > best["val"]:
        print(" New best epoch! ")
        best = {"val": val_auc, "test": test_auc, "epoch": epoch}
        if val_auc>best_auc_global:
            torch.save(model.state_dict(), global_model_path)
            best_auc_global=val_auc
    print()

trial_results.append({
    'type':'qple_linear_per_embed',
    "auc_val": val_auc,
    "auc_test": test_auc,
    "d_token": d_Block,
    "n_blocks": n_Blocks,
    "n_heads": Attention_n_heads,
    "ffn_d_hidden": FFN_d_hidden,
    'best':best,
    'timer':timer.__str__(),
    'train_auc':train_auc
  })

with open(str(RESULTS_DIR / "ftt_qple_linear_periodic_hc.json"), "w") as f:
    json.dump(trial_results, f, indent=4)

## 11. Variant 7 — Vanilla FTT with Lion Optimizer

Swaps AdamW for Lion (EvoLved Sign Momentum), a sign-based optimizer that often matches AdamW with less memory. Same vanilla FTT architecture.

### 11.1 Model

In [ ]:
class FTTransformer(nn.Module):
    def __init__(
        self,
        *,
        n_cont_features: int,
        cat_cardinalities: List[int],
        _is_default: bool = False,
        **backbone_kwargs,
    ) -> None:
        if n_cont_features < 0:
            raise ValueError(
                f'n_cont_features must be non-negative, however: {n_cont_features=}'
            )
        if n_cont_features == 0 and not cat_cardinalities:
            raise ValueError(
                'At least one type of features must be presented, however:'
                f' {n_cont_features=}, {cat_cardinalities=}'
            )
        if 'n_tokens' in backbone_kwargs:
            raise ValueError(
                'backbone_kwargs must not contain key "n_tokens"'
                ' (the number of tokens will be inferred automatically)'
            )

        super().__init__()
        d_block: int = backbone_kwargs['d_block']
        self.cls_embedding = _CLSEmbedding(d_block)

        # >>> Feature embeddings (Figure 2a in the paper).
        # self.cont_embeddings = (
        # LinearSiLUEmbeddings(n_cont_features, d_block) if n_cont_features > 0 else None
        # )
        self.cont_embeddings = (
            LinearEmbeddings(n_cont_features, d_block) if n_cont_features > 0 else None
        )
        self.cat_embeddings = (
            CategoricalEmbeddings(cat_cardinalities, d_block, True)
            if cat_cardinalities
            else None
        )
        # <<<

        self.backbone = FTTransformerBackbone(
            **backbone_kwargs,
            n_tokens=(
                None
                if backbone_kwargs.get('linformer_kv_compression_ratio') is None
                else 1 + n_cont_features + len(cat_cardinalities)
            ),
        )
        self._is_default = _is_default

    @classmethod
    def get_default_kwargs(cls, n_blocks: int = 3) -> Dict[str, Any]:
        """Get the default hyperparameters.

        Args:
            n_blocks: the number of blocks. The supported values are: 1, 2, 3, 4, 5, 6.
        Returns:
            the default keyword arguments for the constructor.
        """
        if n_blocks < 0 or n_blocks > 6:
            raise ValueError(
                'Default configurations are available'
                ' only for the following values of n_blocks: 1, 2, 3, 4, 5, 6.'
                f' However, {n_blocks=}'
            )
        return {
            'n_blocks': n_blocks,
            'd_block': [96, 128, 192, 256, 320, 384][n_blocks - 1],
            'attention_n_heads': 8,
            'attention_dropout': [0.1, 0.15, 0.2, 0.25, 0.3, 0.35][n_blocks - 1],
            'ffn_d_hidden': None,
            # "4 / 3" for ReGLU leads to almost the same number of parameters
            # as "2.0" for ReLU.
            'ffn_d_hidden_multiplier': 4 / 3,
            'ffn_dropout': [0.0, 0.05, 0.1, 0.15, 0.2, 0.25][n_blocks - 1],
            'residual_dropout': 0.0,
            '_is_default': True,
        }

    def make_parameter_groups(self) -> List[Dict[str, Any]]:
        """Make parameter groups for optimizers.

        The difference with calling this method instead of
        `.parameters()` is that this method always sets `weight_decay=0.0`
        for some of the parameters.

        Returns:
            the parameter groups that can be passed to PyTorch optimizers.
        """

        def get_parameters(m: Optional[nn.Module]) -> Iterable[Parameter]:
            return () if m is None else m.parameters()

        zero_wd_group: Dict[str, Any] = {
            'params': set(
                itertools.chain(
                    get_parameters(self.cls_embedding),
                    get_parameters(self.cont_embeddings),
                    get_parameters(self.cat_embeddings),
                    itertools.chain.from_iterable(
                        m.parameters()
                        for block in self.backbone.blocks
                        for name, m in block.named_children()
                        if name.endswith('_normalization')
                    ),
                    (
                        p
                        for name, p in self.named_parameters()
                        if name.endswith('.bias')
                    ),
                )
            ),
            'weight_decay': 0.0,
        }
        main_group: Dict[str, Any] = {
            'params': [p for p in self.parameters() if p not in zero_wd_group['params']]
        }
        zero_wd_group['params'] = list(zero_wd_group['params'])
        return [main_group, zero_wd_group]

    def make_default_optimizer(self) -> torch.optim.AdamW:
        """Create the "default" `torch.nn.AdamW` suitable for the *default* FT-Transformer.

        Returns:
            the optimizer.
        """ # noqa: E501
        if not self._is_default:
            warnings.warn(
                'The default opimizer is supposed to be used in a combination'
                ' with the default FT-Transformer.'
            )
        return torch.optim.AdamW(
            self.make_parameter_groups(), lr=1e-4, weight_decay=1e-5
        )

    _FORWARD_BAD_ARGS_MESSAGE = (
        'Based on the arguments passed to the constructor of FTTransformer, {}'
    )

    def forward(self, x_cont: Optional[Tensor], x_cat: Optional[Tensor]) -> Tensor:
        """Do the forward pass."""
        x_any = x_cat if x_cont is None else x_cont
        if x_any is None:
            raise ValueError('At least one of x_cont and x_cat must be provided.')

        x_embeddings: List[Tensor] = []
        if self.cls_embedding is not None:
            x_embeddings.append(self.cls_embedding(x_any.shape[:-1]))

        for argname, argvalue, module in [
            ('x_cont', x_cont, self.cont_embeddings),
            ('x_cat', x_cat, self.cat_embeddings),
        ]:
            if module is None:
                if argvalue is not None:
                    raise ValueError(
                        FTTransformer._FORWARD_BAD_ARGS_MESSAGE.format(
                            f'{argname} must be None'
                        )
                    )
            else:
                if argvalue is None:
                    raise ValueError(
                        FTTransformer._FORWARD_BAD_ARGS_MESSAGE.format(
                            f'{argname} must not be None'
                        )
                    )
                x_embeddings.append(module(argvalue))
        assert x_embeddings, _INTERNAL_ERROR
        x = torch.cat(x_embeddings, dim=1)
        x = self.backbone(x)
        return x

### 11.2 Train + Evaluate

In [ ]:
%pip install lion_pytorch -q

In [ ]:
!pip install lion_pytorch -q
from lion_pytorch import Lion

trial_results = []
best_auc_global=0.774
global_model_path = str(MODELS_DIR / "ftt_lion_hc.pth")
d_out=1
early_stopping = delu.tools.EarlyStopping(patience, mode="max")
d_Block = 64
n_Blocks = 3
Attention_n_heads = 4
FFN_d_hidden = 2
Lr=2e-4
BATCH_SIZE=1024+4096
# Print trial parameters at the start of each trial
print("\n" + "="*70)
print(" Hyperparameters:")
print(f" d_token = {d_Block}")
print(f" n_blocks = {n_Blocks}")
print(f" attention_n_heads = {Attention_n_heads}")
print(f" ffn_d_hidden = {FFN_d_hidden}")
print("="*70 + "\n")
model = FTTransformer(
    n_cont_features=d_numerical,
    cat_cardinalities=cat_cardinalities_final,
    n_blocks=n_Blocks,
    d_block=d_Block,
    attention_n_heads=Attention_n_heads,
    ffn_d_hidden_multiplier=FFN_d_hidden,
    attention_dropout=0.05,
    ffn_dropout=0.05,
    residual_dropout=0.05,
    d_out=1 # output a single logit (binary)
).to(device)

optimizer = Lion(
        model.parameters(),
        lr=Lr / 10, # Lion needs ~3-10x smaller LR than Adam
        weight_decay=1e-2 # Lion benefits from higher weight decay than Adam
    )

scheduler = torch.optim.lr_scheduler.OneCycleLR(
optimizer,
max_lr=Lr * 5, # Recommended: 3–10× base LR
steps_per_epoch=math.ceil(X_train.shape[0] / BATCH_SIZE), # NUMBER OF BATCHES PER EPOCH
epochs=n_epochs,
pct_start=0.2,
anneal_strategy="cos"
)
batch_size = BATCH_SIZE
epoch_size = math.ceil(X_train.shape[0] / batch_size)
timer = delu.tools.Timer()


n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f" Trainable params: {n_params:,}")

print(f'Test score before training: AUC Val = {evaluate("val"):.4f}')


best = {
    "val": -math.inf,
    "test": -math.inf,
    "epoch": -1,
}

print(f"Device: {device.type.upper()}")
print("-" * 88 + "\n")
timer.run()
for epoch in range(n_epochs):
    pos_weight = torch.tensor(6, device=device)
    neg_weight = torch.tensor(1, device=device)
    for batch in tqdm(
        delu.iter_batches(data["train"], batch_size, shuffle=True),
        desc=f"Epoch {epoch}",
        total=epoch_size,
    ):
        model.train()
        optimizer.zero_grad()

        y_batch = batch["y"].float()
        weights = torch.where(
            y_batch == 1,
            pos_weight,
            neg_weight
        )

        loss = loss_fn(apply_model(batch), batch["y"].float(), weight=weights)
        loss.backward()
        optimizer.step()
        scheduler.step()

    val_auc = evaluate("val")
    test_auc = evaluate("test")
    train_auc = evaluate('train')
    print(f" AUC Train {train_auc:.4f} Val {val_auc:.4f} (test) {test_auc:.4f} [time] {timer}")

    early_stopping.update(val_auc)
    if early_stopping.should_stop():
      print('Early stopping triggered')
      break

    if val_auc > best["val"]:
        print(" New best epoch! ")
        best = {"val": val_auc, "test": test_auc, "epoch": epoch}
        if val_auc>best_auc_global:
            torch.save(model.state_dict(), global_model_path)
            best_auc_global=val_auc
    print()

trial_results.append({
    'type':'Lion Optimizer',
    "auc_val": val_auc,
    "auc_test": test_auc,
    "d_token": d_Block,
    "n_blocks": n_Blocks,
    "n_heads": Attention_n_heads,
    "ffn_d_hidden": FFN_d_hidden,
    'best':best,
    'timer':timer.__str__(),
    'train_auc':train_auc
  })

with open(str(RESULTS_DIR / "ftt_lion_hc.json"), "w") as f:
    json.dump(trial_results, f, indent=4)

## 12. Variant 8 — Sparsemax Attention

Replaces the softmax in self-attention with Sparsemax, which can produce exactly-zero attention weights. The hypothesis is that sparse attention is more interpretable and may regularize the model on small-ish tabular data.

### 12.1 Sparsemax module

In [ ]:
class Sparsemax(nn.Module):
    def __init__(self, dim=-1):
        super(Sparsemax, self).__init__()
        self.dim = dim

    def forward(self, input: Tensor) -> Tensor:
        # Shift for numerical stability
        input = input - torch.max(input, dim=self.dim, keepdim=True)[0]

        z_sorted, _ = torch.sort(input, descending=True, dim=self.dim)
        z_cumsum = torch.cumsum(z_sorted, dim=self.dim)

        k_range = torch.arange(1, input.size(self.dim) + 1, device=input.device)
        k_mask = 1 + k_range * z_sorted > z_cumsum

        # We need the last index where the condition is true
        k_selected = torch.sum(k_mask, dim=self.dim, keepdim=True)
        tau = (torch.gather(z_cumsum, self.dim, k_selected - 1) - 1) / k_selected

        return torch.clamp(input - tau, min=0)

class MultiheadAttentionSparsemax(nn.Module):
    def __init__(
        self,
        *,
        d_embedding: int,
        n_heads: int,
        dropout: float,
        # Linformer arguments.
        n_tokens: Optional[int] = None,
        linformer_kv_compression_ratio: Optional[float] = None,
        linformer_kv_compression_sharing: Optional[
            _LINFORMER_KV_COMPRESSION_SHARING
        ] = None,
    ) -> None:
        if n_heads < 1:
            raise ValueError(f'n_heads must be positive, however: {n_heads=}')
        if d_embedding % n_heads:
            raise ValueError(
                'd_embedding must be a multiple of n_heads,'
                f' however: {d_embedding=}, {n_heads=}'
            )

        super().__init__()
        self.W_q = nn.Linear(d_embedding, d_embedding)
        self.W_k = nn.Linear(d_embedding, d_embedding)
        self.W_v = nn.Linear(d_embedding, d_embedding)
        self.W_out = nn.Linear(d_embedding, d_embedding) if n_heads > 1 else None
        self.dropout = nn.Dropout(dropout) if dropout else None
        self._n_heads = n_heads
        self.sparsemax = Sparsemax(dim=-1)

        if linformer_kv_compression_ratio is not None:
            if n_tokens is None:
                raise ValueError(
                    'If linformer_kv_compression_ratio is not None,'
                    ' then n_tokens also must not be None'
                )
            if linformer_kv_compression_sharing not in typing.get_args(
                _LINFORMER_KV_COMPRESSION_SHARING
            ):
                raise ValueError(
                    'Valid values of linformer_kv_compression_sharing include:'
                    f' {typing.get_args(_LINFORMER_KV_COMPRESSION_SHARING)},'
                    f' however: {linformer_kv_compression_sharing=}'
                )
            if (
                linformer_kv_compression_ratio <= 0.0
                or linformer_kv_compression_ratio >= 1.0
            ):
                raise ValueError(
                    'linformer_kv_compression_ratio must be from the open interval'
                    f' (0.0, 1.0), however: {linformer_kv_compression_ratio=}'
                )

            def make_linformer_kv_compression():
                return nn.Linear(
                    n_tokens,
                    max(int(n_tokens * linformer_kv_compression_ratio), 1),
                    bias=False,
                )

            self.key_compression = make_linformer_kv_compression()
            self.value_compression = (
                make_linformer_kv_compression()
                if linformer_kv_compression_sharing == 'headwise'
                else None
            )
        else:
            if n_tokens is not None:
                raise ValueError(
                    'If linformer_kv_compression_ratio is None,'
                    ' then n_tokens also must be None'
                )
            if linformer_kv_compression_sharing is not None:
                raise ValueError(
                    'If linformer_kv_compression_ratio is None,'
                    ' then linformer_kv_compression_sharing also must be None'
                )
            self.key_compression = None
            self.value_compression = None

        for m in [self.W_q, self.W_k, self.W_v]:
            nn.init.zeros_(m.bias)
        if self.W_out is not None:
            nn.init.zeros_(self.W_out.bias)

    def _reshape(self, x: Tensor) -> Tensor:
        batch_size, n_tokens, d = x.shape
        d_head = d // self._n_heads
        return (
            x.reshape(batch_size, n_tokens, self._n_heads, d_head)
            .transpose(1, 2)
            .reshape(batch_size * self._n_heads, n_tokens, d_head)
        )

    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        """Do the forward pass."""
        q, k, v = self.W_q(x_q), self.W_k(x_kv), self.W_v(x_kv)
        if self.key_compression is not None:
            k = self.key_compression(k.transpose(1, 2)).transpose(1, 2)
            v = (
                self.key_compression
                if self.value_compression is None
                else self.value_compression
            )(v.transpose(1, 2)).transpose(1, 2)

        batch_size = len(q)
        d_head_key = k.shape[-1] // self._n_heads
        d_head_value = v.shape[-1] // self._n_heads
        n_q_tokens = q.shape[1]

        q = self._reshape(q)
        k = self._reshape(k)
        attention_logits = q @ k.transpose(1, 2) / math.sqrt(d_head_key)
        # attention_probs = F.softmax(attention_logits, dim=-1)
        attention_probs = self.sparsemax(attention_logits)
        if self.dropout is not None:
            attention_probs = self.dropout(attention_probs)
        x = attention_probs @ self._reshape(v)
        x = (
            x.reshape(batch_size, self._n_heads, n_q_tokens, d_head_value)
            .transpose(1, 2)
            .reshape(batch_size, n_q_tokens, self._n_heads * d_head_value)
        )
        if self.W_out is not None:
            x = self.W_out(x)
        return x

class _CLSEmbedding(nn.Module):
    def __init__(self, d_embedding: int) -> None:
        super().__init__()
        self.weight = Parameter(torch.empty(d_embedding))
        self.reset_parameters()

    def reset_parameters(self) -> None:
        d_rsqrt = self.weight.shape[-1] ** -0.5
        nn.init.uniform_(self.weight, -d_rsqrt, d_rsqrt)

    def forward(self, batch_dims: Tuple[int]) -> Tensor:
        if not batch_dims:
            raise ValueError('The input must be non-empty')

        return self.weight.expand(*batch_dims, 1, -1)

class FTTransformerBackbone(nn.Module):
    """The backbone of FT-Transformer.

    The differences with Transformer from the paper
    ["Attention Is All You Need"](https://arxiv.org/abs/1706.03762) are as follows:

    - the so called "PreNorm" variation is used
        (`norm_first=True` in terms of `torch.nn.TransformerEncoderLayer`)
    - the very first normalization is skipped. This is **CRUCIAL** for FT-Transformer
        in the PreNorm configuration.
    """

    def __init__(
        self,
        *,
        d_out: Optional[int],
        n_blocks: int,
        d_block: int,
        attention_n_heads: int,
        attention_dropout: float,
        ffn_d_hidden: Optional[int] = None,
        ffn_d_hidden_multiplier: Optional[float],
        ffn_dropout: float,
        # NOTE[DIFF]
        # In the paper, FT-Transformer uses the ReGLU activation.
        # Here, to illustrate the difference, ReLU activation is also supported
        # (in particular, see the docstring).
        ffn_activation: _TransformerFFNActivation = 'ReGLU',
        residual_dropout: float,
        n_tokens: Optional[int] = None,
        linformer_kv_compression_ratio: Optional[float] = None,
        linformer_kv_compression_sharing: Optional[
            _LINFORMER_KV_COMPRESSION_SHARING
        ] = None,
    ):
        # noqa: E501
        if ffn_activation not in typing.get_args(_TransformerFFNActivation):
            raise ValueError(
                'ffn_activation must be one of'
                f' {typing.get_args(_TransformerFFNActivation)}.'
                f' However: {ffn_activation=}'
            )
        if ffn_d_hidden is None:
            if ffn_d_hidden_multiplier is None:
                raise ValueError(
                    'If ffn_d_hidden is None,'
                    ' then ffn_d_hidden_multiplier must not be None'
                )
            ffn_d_hidden = int(d_block * cast(float, ffn_d_hidden_multiplier))
        else:
            if ffn_d_hidden_multiplier is not None:
                raise ValueError(
                    'If ffn_d_hidden is not None,'
                    ' then ffn_d_hidden_multiplier must be None'
                )

        super().__init__()
        ffn_use_reglu = (ffn_activation == 'ReGLU') or (ffn_activation == 'SwiGLU')
        ffn_use_swiglu = ffn_activation == 'SwiGLU'
        self.blocks = nn.ModuleList(
            [
                nn.ModuleDict(
                    {
                        # >>> attention
                        'attention': MultiheadAttentionSparsemax(
                            d_embedding=d_block,
                            n_heads=attention_n_heads,
                            dropout=attention_dropout,
                            n_tokens=n_tokens,
                            linformer_kv_compression_ratio=linformer_kv_compression_ratio,
                            linformer_kv_compression_sharing=linformer_kv_compression_sharing,
                        ),
                        'attention_residual_dropout': nn.Dropout(residual_dropout),
                        # >>> feed-forward
                        'ffn_normalization': nn.LayerNorm(d_block),
                        'ffn': _named_sequential(
                            (
                                'linear1',
                                # ReGLU divides dimension by 2,
                                # so multiplying by 2 to compensate for this.
                                nn.Linear(
                                    d_block, ffn_d_hidden * (2 if ffn_use_reglu else 1)
                                ),
                            ),
                            ('activation', _SwiGLU() if ffn_use_swiglu else _ReGLU() if ffn_use_reglu else nn.ReLU()),
                            ('dropout', nn.Dropout(ffn_dropout)),
                            ('linear2', nn.Linear(ffn_d_hidden, d_block)),
                        ),
                        'ffn_residual_dropout': nn.Dropout(residual_dropout),
                        # >>> output (for hook-based introspection)
                        'output': nn.Identity(),
                        # >>> the very first normalization
                        **(
                            {}
                            if layer_idx == 0
                            else {'attention_normalization': nn.LayerNorm(d_block)}
                        ),
                    }
                )
                for layer_idx in range(n_blocks)
            ]
        )
        self.output = (
            None
            if d_out is None
            else _named_sequential(
                ('normalization', nn.LayerNorm(d_block)),
                ('activation', nn.ReLU()),
                ('linear', nn.Linear(d_block, d_out)),
            )
        )

    def forward(self, x: Tensor) -> Tensor:
        """Do the forward pass."""
        if x.ndim != 3:
            raise ValueError(
                f'The input must have exactly three dimension, however: {x.ndim=}'
            )

        n_blocks = len(self.blocks)
        for i_block, block in enumerate(self.blocks):
            block = cast(nn.ModuleDict, block)

            x_identity = x
            if 'attention_normalization' in block:
                x = block['attention_normalization'](x)
            x = block['attention'](x[:, :1] if i_block + 1 == n_blocks else x, x)
            x = block['attention_residual_dropout'](x)
            x = x_identity + x

            x_identity = x
            x = block['ffn_normalization'](x)
            x = block['ffn'](x)
            x = block['ffn_residual_dropout'](x)
            x = x_identity + x

            x = block['output'](x)

        x = x[:, 0] # The representation of [CLS]-token.

        if self.output is not None:
            x = self.output(x)
        return x

### 12.2 Model

In [ ]:
class FTTransformer(nn.Module):
    def __init__(
        self,
        *,
        n_cont_features: int,
        cat_cardinalities: List[int],
        _is_default: bool = False,
        **backbone_kwargs,
    ) -> None:
        if n_cont_features < 0:
            raise ValueError(
                f'n_cont_features must be non-negative, however: {n_cont_features=}'
            )
        if n_cont_features == 0 and not cat_cardinalities:
            raise ValueError(
                'At least one type of features must be presented, however:'
                f' {n_cont_features=}, {cat_cardinalities=}'
            )
        if 'n_tokens' in backbone_kwargs:
            raise ValueError(
                'backbone_kwargs must not contain key "n_tokens"'
                ' (the number of tokens will be inferred automatically)'
            )

        super().__init__()
        d_block: int = backbone_kwargs['d_block']
        self.cls_embedding = _CLSEmbedding(d_block)

        # >>> Feature embeddings (Figure 2a in the paper).
        # self.cont_embeddings = (
        # LinearSiLUEmbeddings(n_cont_features, d_block) if n_cont_features > 0 else None
        # )
        self.cont_embeddings = (
            LinearEmbeddings(n_cont_features, d_block) if n_cont_features > 0 else None
        )
        self.cat_embeddings = (
            CategoricalEmbeddings(cat_cardinalities, d_block, True)
            if cat_cardinalities
            else None
        )
        # <<<

        self.backbone = FTTransformerBackbone(
            **backbone_kwargs,
            n_tokens=(
                None
                if backbone_kwargs.get('linformer_kv_compression_ratio') is None
                else 1 + n_cont_features + len(cat_cardinalities)
            ),
        )
        self._is_default = _is_default

    @classmethod
    def get_default_kwargs(cls, n_blocks: int = 3) -> Dict[str, Any]:
        """Get the default hyperparameters.

        Args:
            n_blocks: the number of blocks. The supported values are: 1, 2, 3, 4, 5, 6.
        Returns:
            the default keyword arguments for the constructor.
        """
        if n_blocks < 0 or n_blocks > 6:
            raise ValueError(
                'Default configurations are available'
                ' only for the following values of n_blocks: 1, 2, 3, 4, 5, 6.'
                f' However, {n_blocks=}'
            )
        return {
            'n_blocks': n_blocks,
            'd_block': [96, 128, 192, 256, 320, 384][n_blocks - 1],
            'attention_n_heads': 8,
            'attention_dropout': [0.1, 0.15, 0.2, 0.25, 0.3, 0.35][n_blocks - 1],
            'ffn_d_hidden': None,
            # "4 / 3" for ReGLU leads to almost the same number of parameters
            # as "2.0" for ReLU.
            'ffn_d_hidden_multiplier': 4 / 3,
            'ffn_dropout': [0.0, 0.05, 0.1, 0.15, 0.2, 0.25][n_blocks - 1],
            'residual_dropout': 0.0,
            '_is_default': True,
        }

    def make_parameter_groups(self) -> List[Dict[str, Any]]:
        """Make parameter groups for optimizers.

        The difference with calling this method instead of
        `.parameters()` is that this method always sets `weight_decay=0.0`
        for some of the parameters.

        Returns:
            the parameter groups that can be passed to PyTorch optimizers.
        """

        def get_parameters(m: Optional[nn.Module]) -> Iterable[Parameter]:
            return () if m is None else m.parameters()

        zero_wd_group: Dict[str, Any] = {
            'params': set(
                itertools.chain(
                    get_parameters(self.cls_embedding),
                    get_parameters(self.cont_embeddings),
                    get_parameters(self.cat_embeddings),
                    itertools.chain.from_iterable(
                        m.parameters()
                        for block in self.backbone.blocks
                        for name, m in block.named_children()
                        if name.endswith('_normalization')
                    ),
                    (
                        p
                        for name, p in self.named_parameters()
                        if name.endswith('.bias')
                    ),
                )
            ),
            'weight_decay': 0.0,
        }
        main_group: Dict[str, Any] = {
            'params': [p for p in self.parameters() if p not in zero_wd_group['params']]
        }
        zero_wd_group['params'] = list(zero_wd_group['params'])
        return [main_group, zero_wd_group]

    def make_default_optimizer(self) -> torch.optim.AdamW:
        """Create the "default" `torch.nn.AdamW` suitable for the *default* FT-Transformer.

        Returns:
            the optimizer.
        """ # noqa: E501
        if not self._is_default:
            warnings.warn(
                'The default opimizer is supposed to be used in a combination'
                ' with the default FT-Transformer.'
            )
        return torch.optim.AdamW(
            self.make_parameter_groups(), lr=1e-4, weight_decay=1e-5
        )

    _FORWARD_BAD_ARGS_MESSAGE = (
        'Based on the arguments passed to the constructor of FTTransformer, {}'
    )

    def forward(self, x_cont: Optional[Tensor], x_cat: Optional[Tensor]) -> Tensor:
        """Do the forward pass."""
        x_any = x_cat if x_cont is None else x_cont
        if x_any is None:
            raise ValueError('At least one of x_cont and x_cat must be provided.')

        x_embeddings: List[Tensor] = []
        if self.cls_embedding is not None:
            x_embeddings.append(self.cls_embedding(x_any.shape[:-1]))

        for argname, argvalue, module in [
            ('x_cont', x_cont, self.cont_embeddings),
            ('x_cat', x_cat, self.cat_embeddings),
        ]:
            if module is None:
                if argvalue is not None:
                    raise ValueError(
                        FTTransformer._FORWARD_BAD_ARGS_MESSAGE.format(
                            f'{argname} must be None'
                        )
                    )
            else:
                if argvalue is None:
                    raise ValueError(
                        FTTransformer._FORWARD_BAD_ARGS_MESSAGE.format(
                            f'{argname} must not be None'
                        )
                    )
                x_embeddings.append(module(argvalue))
        assert x_embeddings, _INTERNAL_ERROR
        x = torch.cat(x_embeddings, dim=1)
        x = self.backbone(x)
        return x

### 12.3 Train + Evaluate

In [ ]:
trial_results = []
best_auc_global=0.774
global_model_path = str(MODELS_DIR / "ftt_sparsemax_hc.pth")
d_out=1
early_stopping = delu.tools.EarlyStopping(patience, mode="max")
d_Block = 64
n_Blocks = 3
Attention_n_heads = 4
FFN_d_hidden = 2
Lr=2e-4
BATCH_SIZE=2048
# Print trial parameters at the start of each trial
print("\n" + "="*70)
print(" Hyperparameters:")
print(f" d_token = {d_Block}")
print(f" n_blocks = {n_Blocks}")
print(f" attention_n_heads = {Attention_n_heads}")
print(f" ffn_d_hidden = {FFN_d_hidden}")
print("="*70 + "\n")
model = FTTransformer(
    n_cont_features=d_numerical,
    cat_cardinalities=cat_cardinalities_final,
    n_blocks=n_Blocks,
    d_block=d_Block,
    attention_n_heads=Attention_n_heads,
    ffn_d_hidden_multiplier=FFN_d_hidden,
    attention_dropout=0.05,
    ffn_dropout=0.05,
    residual_dropout=0.05,
    d_out=1 # output a single logit (binary)
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=Lr, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
optimizer,
max_lr=Lr * 5, # Recommended: 3–10× base LR
steps_per_epoch=math.ceil(X_train.shape[0] / BATCH_SIZE), # NUMBER OF BATCHES PER EPOCH
epochs=n_epochs,
pct_start=0.2,
anneal_strategy="cos"
)
batch_size = BATCH_SIZE
epoch_size = math.ceil(X_train.shape[0] / batch_size)
timer = delu.tools.Timer()


n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f" Trainable params: {n_params:,}")

print(f'Test score before training: AUC Val = {evaluate("val"):.4f}')


best = {
    "val": -math.inf,
    "test": -math.inf,
    "epoch": -1,
}

print(f"Device: {device.type.upper()}")
print("-" * 88 + "\n")
timer.run()
for epoch in range(n_epochs):
    pos_weight = torch.tensor(6, device=device)
    neg_weight = torch.tensor(1, device=device)
    for batch in tqdm(
        delu.iter_batches(data["train"], batch_size, shuffle=True),
        desc=f"Epoch {epoch}",
        total=epoch_size,
    ):
        model.train()
        optimizer.zero_grad()

        y_batch = batch["y"].float()
        weights = torch.where(
            y_batch == 1,
            pos_weight,
            neg_weight
        )

        loss = loss_fn(apply_model(batch), batch["y"].float(), weight=weights)
        loss.backward()
        optimizer.step()
        scheduler.step()

    val_auc = evaluate("val")
    test_auc = evaluate("test")
    train_auc = evaluate('train')
    print(f" AUC Train {train_auc:.4f} Val {val_auc:.4f} (test) {test_auc:.4f} [time] {timer}")

    early_stopping.update(val_auc)
    if early_stopping.should_stop():
      print('Early stopping triggered')
      break

    if val_auc > best["val"]:
        print(" New best epoch! ")
        best = {"val": val_auc, "test": test_auc, "epoch": epoch}
        if val_auc>best_auc_global:
            torch.save(model.state_dict(), global_model_path)
            best_auc_global=val_auc
    print()

trial_results.append({
    'type':'sparsemax_attn',
    "auc_val": val_auc,
    "auc_test": test_auc,
    "d_token": d_Block,
    "n_blocks": n_Blocks,
    "n_heads": Attention_n_heads,
    "ffn_d_hidden": FFN_d_hidden,
    'best':best,
    'timer':timer.__str__(),
    'train_auc':train_auc
  })

with open(str(RESULTS_DIR / "ftt_sparsemax_hc.json"), "w") as f:
    json.dump(trial_results, f, indent=4)

## 13. Variant 9 — Pairwise AUC Loss

Replaces BCE with a differentiable pairwise AUC surrogate. The loss directly optimizes the ranking of positives over negatives via random pair sampling — closer in spirit to what AUC actually measures.

### 13.1 Loss

In [ ]:
class PairwiseAUCLoss(nn.Module):
    """
    Pairwise AUC surrogate loss — Adam compatible.
    Directly optimizes ranking of positives over negatives.
    """
    def __init__(self, gamma: float = 1.0, n_pairs: int = 2048):
        super().__init__()
        self.gamma = gamma
        self.n_pairs = n_pairs

    def forward(self, logits: Tensor, labels: Tensor) -> Tensor:
        pos_scores = logits[labels == 1]
        neg_scores = logits[labels == 0]

        if pos_scores.numel() == 0 or neg_scores.numel() == 0:
            return torch.tensor(0.0, requires_grad=True, device=logits.device)

        n = min(self.n_pairs, pos_scores.numel() * neg_scores.numel())
        pos_idx = torch.randint(0, pos_scores.numel(), (n,), device=logits.device)
        neg_idx = torch.randint(0, neg_scores.numel(), (n,), device=logits.device)

        diff = pos_scores[pos_idx] - neg_scores[neg_idx]
        return torch.sigmoid(-self.gamma * diff).mean()


class AnnealedAUCLoss(nn.Module):
    """
    BCE + PairwiseAUC with annealing auc_weight over training.
    BCE stabilizes early training, AUC loss refines ranking later.
    Compatible with your existing weighted BCE setup.
    """
    def __init__(self, gamma: float = 1.0, n_pairs: int = 2048):
        super().__init__()
        self.gamma = gamma
        self.n_pairs = n_pairs
        self.auc_weight = 0.0 # updated externally each epoch
        self.pairwise = PairwiseAUCLoss(gamma=gamma, n_pairs=n_pairs)

    def forward(
        self,
        logits: Tensor,
        labels: Tensor,
        weight: Tensor = None
    ) -> Tensor:
        bce = F.binary_cross_entropy_with_logits(
            logits, labels.float(), weight=weight
        )
        auc = self.pairwise(logits, labels)
        return (1 - self.auc_weight) * bce + self.auc_weight * auc

### 13.2 Train + Evaluate

In [ ]:
trial_results = []
best_auc_global=0.774
global_model_path = str(MODELS_DIR / "ftt_pairwise_auc_hc.pth")
d_out=1
early_stopping = delu.tools.EarlyStopping(patience, mode="max")
d_Block = 64
n_Blocks = 3
Attention_n_heads = 4
FFN_d_hidden = 2
Lr=2e-4
BATCH_SIZE=4096
# Print trial parameters at the start of each trial
print("\n" + "="*70)
print(" Hyperparameters:")
print(f" d_token = {d_Block}")
print(f" n_blocks = {n_Blocks}")
print(f" attention_n_heads = {Attention_n_heads}")
print(f" ffn_d_hidden = {FFN_d_hidden}")
print("="*70 + "\n")
model = FTTransformer(
    n_cont_features=d_numerical,
    cat_cardinalities=cat_cardinalities_final,
    n_blocks=n_Blocks,
    d_block=d_Block,
    boundaries = boundaries_df,
    attention_n_heads=Attention_n_heads,
    ffn_d_hidden_multiplier=FFN_d_hidden,
    # ffn_activation='SwiGLU',
    attention_dropout=0.05,
    ffn_dropout=0.05,
    residual_dropout=0.05,
    d_out=1 # output a single logit (binary)
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=Lr, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
optimizer,
max_lr=Lr * 5, # Recommended: 3–10× base LR
steps_per_epoch=math.ceil(X_train.shape[0] / BATCH_SIZE), # NUMBER OF BATCHES PER EPOCH
epochs=n_epochs,
pct_start=0.2,
anneal_strategy="cos"
)
batch_size = BATCH_SIZE
epoch_size = math.ceil(X_train.shape[0] / batch_size)
timer = delu.tools.Timer()
loss_fn = AnnealedAUCLoss(gamma=1.0, n_pairs=2048)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f" Trainable params: {n_params:,}")

print(f'Test score before training: AUC Val = {evaluate("val"):.4f}')


best = {
    "val": -math.inf,
    "test": -math.inf,
    "epoch": -1,
}

print(f"Device: {device.type.upper()}")
print("-" * 88 + "\n")
timer.run()
for epoch in range(n_epochs):
    pos_weight = torch.tensor(6, device=device)
    neg_weight = torch.tensor(1, device=device)
    for batch in tqdm(
        delu.iter_batches(data["train"], batch_size, shuffle=True),
        desc=f"Epoch {epoch}",
        total=epoch_size,
    ):
        model.train()
        optimizer.zero_grad()

        y_batch = batch["y"].float()
        weights = torch.where(
            y_batch == 1,
            pos_weight,
            neg_weight
        )

        loss = loss_fn(apply_model(batch), batch["y"].float(), weight=weights)
        loss.backward()
        optimizer.step()
        scheduler.step()

    val_auc = evaluate("val")
    test_auc = evaluate("test")
    train_auc = evaluate('train')
    print(f" AUC Train {train_auc:.4f} Val {val_auc:.4f} (test) {test_auc:.4f} [time] {timer}")

    early_stopping.update(val_auc)
    if early_stopping.should_stop():
      print('Early stopping triggered')
      break

    if val_auc > best["val"]:
        print(" New best epoch! ")
        best = {"val": val_auc, "test": test_auc, "epoch": epoch}
        if val_auc>best_auc_global:
            torch.save(model.state_dict(), global_model_path)
            best_auc_global=val_auc
    print()

trial_results.append({
    'type':'custom_loss',
    "auc_val": val_auc,
    "auc_test": test_auc,
    "d_token": d_Block,
    "n_blocks": n_Blocks,
    "n_heads": Attention_n_heads,
    "ffn_d_hidden": FFN_d_hidden,
    'best':best,
    'timer':timer.__str__(),
    'train_auc':train_auc
  })

with open(str(RESULTS_DIR / "ftt_pairwise_auc_hc.json"), "w") as f:
    json.dump(trial_results, f, indent=4)

## 14. Variant 10 — MLP-Based Contextual Embedding

A small MLP mixes all numerical features into a context vector, which is then combined with each per-feature value to produce the token. This gives a learnable, more expressive alternative to the cross-feature SwiGLU tokenizer in Variant 3.

### 14.1 Model

In [ ]:
class ContextualContEmbedding(nn.Module):
    def __init__(self, n_cont_features, d_token):
        super().__init__()
        self.n_cont_features = n_cont_features
        self.d_token = d_token
        # A small "bottleneck" MLP to mix feature information
        # This allows the model to learn its own "rotation"
        self.mixer = nn.Sequential(
            nn.Linear(n_cont_features, n_cont_features * 4),
            nn.ReLU(),
            nn.Dropout(0.05),
            nn.Linear(n_cont_features * 4, n_cont_features * d_token)
        )
        # We add a LayerNorm to keep the tokens stable before the Transformer
        self.norm = nn.LayerNorm(d_token)

    def forward(self, x_cont):
        # x_cont shape: [B, n_features]
        B = x_cont.shape[0]
        # 1. Generate mixed tokens
        # Output is a flat vector of [B, n_features * d_token]
        mixed_flat = self.mixer(x_cont)
        # 2. Reshape into tokens for the Transformer backbone
        # Output shape: [B, n_features, d_token]
        tokens = mixed_flat.view(B, self.n_cont_features, self.d_token)
        return self.norm(tokens)

class FTTransformer(nn.Module):
    """The FT-Transformer model from Section 3.3 in the paper."""

    def __init__(
        self,
        *,
        n_cont_features: int,
        cat_cardinalities: List[int],
        boundaries: Optional[Tensor] = None,
        _is_default: bool = False,
        **backbone_kwargs,
    ) -> None:
        if n_cont_features < 0:
            raise ValueError(
                f'n_cont_features must be non-negative, however: {n_cont_features=}'
            )
        if n_cont_features == 0 and not cat_cardinalities:
            raise ValueError(
                'At least one type of features must be presented, however:'
                f' {n_cont_features=}, {cat_cardinalities=}'
            )
        if 'n_tokens' in backbone_kwargs:
            raise ValueError(
                'backbone_kwargs must not contain key "n_tokens"'
                ' (the number of tokens will be inferred automatically)'
            )

        super().__init__()
        d_block: int = backbone_kwargs['d_block']
        self.cls_embedding = _CLSEmbedding(d_block)

        #>>> Feature embeddings (Figure 2a in the paper)
        if n_cont_features > 0:
            self.cont_embeddings = ContextualContEmbedding(n_cont_features, d_block)

        # self.cont_embeddings = (
        # LinearEmbeddings(n_cont_features, d_block) if n_cont_features > 0 else None
        # )
        else: None

        self.cat_embeddings = (
            CategoricalEmbeddings(cat_cardinalities, d_block, True)
            if cat_cardinalities
            else None
        )
        # <<<

        self.backbone = FTTransformerBackbone(
            **backbone_kwargs,
            n_tokens=(
                None
                if backbone_kwargs.get('linformer_kv_compression_ratio') is None
                else 1 + n_cont_features + len(cat_cardinalities)
            ),
        )
        self._is_default = _is_default

    @classmethod
    def get_default_kwargs(cls, n_blocks: int = 3) -> Dict[str, Any]:
        """Get the default hyperparameters.

        Args:
            n_blocks: the number of blocks. The supported values are: 1, 2, 3, 4, 5, 6.
        Returns:
            the default keyword arguments for the constructor.
        """
        if n_blocks < 0 or n_blocks > 6:
            raise ValueError(
                'Default configurations are available'
                ' only for the following values of n_blocks: 1, 2, 3, 4, 5, 6.'
                f' However, {n_blocks=}'
            )
        return {
            'n_blocks': n_blocks,
            'd_block': [96, 128, 192, 256, 320, 384][n_blocks - 1],
            'attention_n_heads': 8,
            'attention_dropout': [0.1, 0.15, 0.2, 0.25, 0.3, 0.35][n_blocks - 1],
            'ffn_d_hidden': None,
            # "4 / 3" for ReGLU leads to almost the same number of parameters
            # as "2.0" for ReLU.
            'ffn_d_hidden_multiplier': 4 / 3,
            'ffn_dropout': [0.0, 0.05, 0.1, 0.15, 0.2, 0.25][n_blocks - 1],
            'residual_dropout': 0.0,
            '_is_default': True,
        }

    def make_parameter_groups(self) -> List[Dict[str, Any]]:
        def get_parameters(m: Optional[nn.Module]) -> Iterable[Parameter]:
            return () if m is None else m.parameters()

        zero_wd_group: Dict[str, Any] = {
            'params': set(
                itertools.chain(
                    get_parameters(self.cls_embedding),
                    get_parameters(self.cont_embeddings),
                    get_parameters(self.cat_embeddings),
                    itertools.chain.from_iterable(
                        m.parameters()
                        for block in self.backbone.blocks
                        for name, m in block.named_children()
                        if name.endswith('_normalization')
                    ),
                    (
                        p
                        for name, p in self.named_parameters()
                        if name.endswith('.bias')
                    ),
                )
            ),
            'weight_decay': 0.0,
        }
        main_group: Dict[str, Any] = {
            'params': [p for p in self.parameters() if p not in zero_wd_group['params']]
        }
        zero_wd_group['params'] = list(zero_wd_group['params'])
        return [main_group, zero_wd_group]

    def make_default_optimizer(self) -> torch.optim.AdamW:
        """Create the "default" `torch.nn.AdamW` suitable for the *default* FT-Transformer.

        Returns:
            the optimizer.
        """ # noqa: E501
        if not self._is_default:
            warnings.warn(
                'The default opimizer is supposed to be used in a combination'
                ' with the default FT-Transformer.'
            )
        return torch.optim.AdamW(
            self.make_parameter_groups(), lr=1e-4, weight_decay=1e-5
        )

    _FORWARD_BAD_ARGS_MESSAGE = (
        'Based on the arguments passed to the constructor of FTTransformer, {}'
    )

    def forward(self, x_cont: Optional[Tensor], x_cat: Optional[Tensor]) -> Tensor:
        """Do the forward pass."""
        x_any = x_cat if x_cont is None else x_cont
        if x_any is None:
            raise ValueError('At least one of x_cont and x_cat must be provided.')

        x_embeddings: List[Tensor] = []
        if self.cls_embedding is not None:
            x_embeddings.append(self.cls_embedding(x_any.shape[:-1]))

        for argname, argvalue, module in [
            ('x_cont', x_cont, self.cont_embeddings),
            ('x_cat', x_cat, self.cat_embeddings),
        ]:
            if module is None:
                if argvalue is not None:
                    raise ValueError(
                        FTTransformer._FORWARD_BAD_ARGS_MESSAGE.format(
                            f'{argname} must be None'
                        )
                    )
            else:
                if argvalue is None:
                    raise ValueError(
                        FTTransformer._FORWARD_BAD_ARGS_MESSAGE.format(
                            f'{argname} must not be None'
                        )
                    )
                x_embeddings.append(module(argvalue))
        assert x_embeddings, _INTERNAL_ERROR
        x = torch.cat(x_embeddings, dim=1)
        x = self.backbone(x)
        return x

### 14.2 Train + Evaluate

In [ ]:
trial_results = []
best_auc_global=0.774
global_model_path = str(MODELS_DIR / "ftt_mlp_embed_hc.pth")
d_out=1
early_stopping = delu.tools.EarlyStopping(patience, mode="max")
d_Block = 64
n_Blocks = 3
Attention_n_heads = 4
FFN_d_hidden = 2
Lr=2e-4
BATCH_SIZE=2048+2048
# Print trial parameters at the start of each trial
print("\n" + "="*70)
print(" Hyperparameters:")
print(f" d_token = {d_Block}")
print(f" n_blocks = {n_Blocks}")
print(f" attention_n_heads = {Attention_n_heads}")
print(f" ffn_d_hidden = {FFN_d_hidden}")
print("="*70 + "\n")
model = FTTransformer(
    n_cont_features=d_numerical,
    cat_cardinalities=cat_cardinalities_final,
    n_blocks=n_Blocks,
    d_block=d_Block,
    attention_n_heads=Attention_n_heads,
    ffn_d_hidden_multiplier=FFN_d_hidden,
    attention_dropout=0.05,
    ffn_dropout=0.05,
    residual_dropout=0.05,
    d_out=1 # output a single logit (binary)
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=Lr, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
optimizer,
max_lr=Lr * 5, # Recommended: 3–10× base LR
steps_per_epoch=math.ceil(X_train.shape[0] / BATCH_SIZE), # NUMBER OF BATCHES PER EPOCH
epochs=n_epochs,
pct_start=0.2,
anneal_strategy="cos"
)
batch_size = BATCH_SIZE
epoch_size = math.ceil(X_train.shape[0] / batch_size)
timer = delu.tools.Timer()


n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f" Trainable params: {n_params:,}")

print(f'Test score before training: AUC Val = {evaluate("val"):.4f}')


best = {
    "val": -math.inf,
    "test": -math.inf,
    "epoch": -1,
}

print(f"Device: {device.type.upper()}")
print("-" * 88 + "\n")
timer.run()
for epoch in range(n_epochs):
    pos_weight = torch.tensor(6, device=device)
    neg_weight = torch.tensor(1, device=device)
    for batch in tqdm(
        delu.iter_batches(data["train"], batch_size, shuffle=True),
        desc=f"Epoch {epoch}",
        total=epoch_size,
    ):
        model.train()
        optimizer.zero_grad()

        y_batch = batch["y"].float()
        weights = torch.where(
            y_batch == 1,
            pos_weight,
            neg_weight
        )

        loss = loss_fn(apply_model(batch), batch["y"].float(), weight=weights)
        loss.backward()
        optimizer.step()
        scheduler.step()

    val_auc = evaluate("val")
    test_auc = evaluate("test")
    train_auc = evaluate('train')
    print(f" AUC Train {train_auc:.4f} Val {val_auc:.4f} (test) {test_auc:.4f} [time] {timer}")

    early_stopping.update(val_auc)
    if early_stopping.should_stop():
      print('Early stopping triggered')
      break

    if val_auc > best["val"]:
        print(" New best epoch! ")
        best = {"val": val_auc, "test": test_auc, "epoch": epoch}
        if val_auc>best_auc_global:
            torch.save(model.state_dict(), global_model_path)
            best_auc_global=val_auc
    print()

trial_results.append({
    'type':'mlp_embed',
    "auc_val": val_auc,
    "auc_test": test_auc,
    "d_token": d_Block,
    "n_blocks": n_Blocks,
    "n_heads": Attention_n_heads,
    "ffn_d_hidden": FFN_d_hidden,
    'best':best,
    'timer':timer.__str__(),
    'train_auc':train_auc
  })

with open(str(RESULTS_DIR / "ftt_mlp_embed_hc.json"), "w") as f:
    json.dump(trial_results, f, indent=4)